<a href="https://colab.research.google.com/github/sulucay01/multimodal-rs-segmentation/blob/dev/notebooks/05_phase3_ablations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Phase 3 ablations

DI725 Term Project — Multimodal Fusion for Remote Sensing Land Cover Segmentation

Three ablation studies and a final combined model. Each ablation varies one component of the multimodal architecture from `04_multimodal.ipynb` while keeping everything else fixed, so any change in test mIoU isolates the contribution of that component.

**Ablation 1 — Text encoder pretraining.** Swaps the frozen CLIP text encoder for RemoteCLIP, a CLIP variant fine-tuned on remote-sensing image-text pairs. Tests whether domain-adapted text representations improve segmentation. All 5 caption variants from Phase 2 are evaluated.

**Ablation 2 — Loss function.** Compares the weighted cross-entropy used in Phase 2 against three region-based alternatives paired with CE: Dice, Lovász-Softmax (a Jaccard / IoU surrogate), and Tversky with asymmetric weighting biased toward recall. Tests whether losses better aligned with the mIoU evaluation metric help.

**Ablation 3 — Fusion mechanism.** Compares the pooled cross-attention used in Phase 2 against FiLM (no spatial selectivity) and multi-token cross-attention (finest spatial selectivity). Tests how the spatial granularity of text-vision fusion affects segmentation.

**Final model.** Combines the strongest single-component improvements from each ablation and tests whether the gains stack.

Reference points: the vision-only UNetFormer baseline at test mIoU 0.6488 (from `03_baseline_models.ipynb`) and the multimodal model at test mIoU 0.6970 (from `04_multimodal.ipynb`, CLIP + `text_qwen3-4b` + pooled cross-attention + weighted CE). Both are loaded from saved JSON files.

All experiments use pixel-level mIoU as the primary metric. Training hyperparameters match Phase 2 (30 epochs, AdamW lr=6e-4, weight decay 0.01, batch size 8, weighted CE + 0.4 × auxiliary head loss, seed 42).

## 1. Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies. open_clip_torch provides the frozen CLIP text encoder.
# huggingface_hub is needed to download RemoteCLIP weights for Ablation 1.
!pip install transformers timm einops wandb open_clip_torch huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00


In [3]:
# Project paths (Drive) and a local working copy for fast I/O during training
from pathlib import Path
import shutil

PROJECT_DIR      = Path('/content/drive/MyDrive/DI725_Project')
DATA_DIR_DRIVE   = PROJECT_DIR / 'data'
SPLITS_CSV       = DATA_DIR_DRIVE / 'captions_with_splits.csv'
CHECKPOINTS_DIR  = PROJECT_DIR / 'checkpoints'
RESULTS_DIR      = PROJECT_DIR / 'results'
REMOTECLIP_CACHE = PROJECT_DIR / 'remoteclip_cache'

CHECKPOINTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
REMOTECLIP_CACHE.mkdir(exist_ok=True)

# Local SSD copy avoids Drive I/O bottleneck during training
LOCAL_DATA        = Path('/content/data')
LOCAL_IMAGES      = LOCAL_DATA / 'images'
LOCAL_MASKS_CLASS = LOCAL_DATA / 'masks_class'

if not LOCAL_DATA.exists():
    print('Copying data to local SSD...')
    shutil.copytree(DATA_DIR_DRIVE / 'images', LOCAL_IMAGES)
    shutil.copytree(DATA_DIR_DRIVE / 'masks_class', LOCAL_MASKS_CLASS)
    print('Done.')
else:
    print('Local data already present.')

print(f'Images: {LOCAL_IMAGES}')
print(f'Masks : {LOCAL_MASKS_CLASS}')

Copying data to local SSD...
Done.
Images: /content/data/images
Masks : /content/data/masks_class


In [4]:
# Imports used across all sections
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Weights & Biases setup for experiment tracking
import wandb
wandb.login()
WANDB_PROJECT = 'di725_project'

Device: cuda


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: selingoktas98 (selingoktas98-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
# Class definitions (same across all project notebooks)
CLASS_NAMES = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
NUM_CLASSES = len(CLASS_NAMES)

# Canonical RGB palette from notebook 01, used by the qualitative figure in Section 10
CLASS_RGB = {
    'Tree':     (  0, 100,   0),
    'Shrub':    (255, 182, 193),
    'Grass':    (154, 205,  50),
    'Crop':     (255, 215,   0),
    'Built-up': (139,  69,  19),
    'Barren':   (211, 211, 211),
    'Water':    (  0,   0, 255),
}

# 5 caption variants from the dataset
CAPTION_COLS = [
    'hybrid_gemma3-4b',
    'hybrid_qwen3-vl-8b',
    'text_qwen3-4b',
    'vision_gemma3-4b',
    'vision_qwen3-vl-8b',
]

# Best caption variant from Phase 2 (used for single-caption ablations in Sections 7-8)
BEST_CAPTION = 'text_qwen3-4b'

# Load split CSV from Drive
split_df = pd.read_csv(SPLITS_CSV)
train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
val_df   = split_df[split_df['split'] == 'val'].reset_index(drop=True)
test_df  = split_df[split_df['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')

# Class weights via median frequency balancing (computed from the training split)
class_avg    = train_df[CLASS_NAMES].mean().values
class_freq   = class_avg / class_avg.sum()
median_freq  = np.median(class_freq)
class_weights = median_freq / (class_freq + 1e-8)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f'Class weights: {class_weights.round(2)}')

Train: 7000  |  Val: 1500  |  Test: 1500
Class weights: [0.15 5.12 0.09 0.24 3.77 1.   2.54]


In [6]:
# Reference results from earlier project notebooks, loaded from saved JSON files
with open(RESULTS_DIR / 'baselines_results.json') as f:
    baseline_data = json.load(f)

with open(RESULTS_DIR / 'multimodal_results.json') as f:
    multimodal_data = json.load(f)

# Vision-only UNetFormer baseline (from notebook 03)
BASELINE_MIOU = baseline_data['unetformer']['test_miou']
BASELINE_OA   = baseline_data['unetformer']['test_oa']
BASELINE_IOUS = baseline_data['unetformer']['class_ious']

# Multimodal model from notebook 04: CLIP + pooled cross-attention + weighted CE on text_qwen3-4b.
# Used as the reference point for Phase 3 ablation comparisons.
MULTIMODAL_MIOU = multimodal_data['multimodal'][BEST_CAPTION]['test_miou']
MULTIMODAL_OA   = multimodal_data['multimodal'][BEST_CAPTION]['test_oa']
MULTIMODAL_IOUS = multimodal_data['multimodal'][BEST_CAPTION]['class_ious']

print(f'Vision-only baseline:         test mIoU {BASELINE_MIOU:.4f}')
print(f'Multimodal reference ({BEST_CAPTION}): test mIoU {MULTIMODAL_MIOU:.4f}')

Vision-only baseline:         test mIoU 0.6488
Multimodal reference (text_qwen3-4b): test mIoU 0.6970


## 2. Dataset and DataLoaders

Returns (image, mask, caption) tuples. The caption column is a constructor argument so the same class works for any of the 5 caption variants. Same as `04_multimodal.ipynb`.

In [7]:
class RSMultiModalDataset(Dataset):
    """Reads (image, class-index mask, caption) tuples.

    Captions are read from a CSV column named in `caption_col`, allowing the same
    Dataset class to serve any of the 5 caption variants.
    """

    def __init__(self, dataframe, images_dir, masks_dir, caption_col):
        self.df          = dataframe.reset_index(drop=True)
        self.images_dir  = Path(images_dir)
        self.masks_dir   = Path(masks_dir)
        self.caption_col = caption_col
        # ImageNet normalization (matches the swsl_resnet18 backbone pretraining)
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row['filename']

        img = Image.open(self.images_dir / fname).convert('RGB')
        img = transforms.functional.to_tensor(img)
        img = self.normalize(img)

        mask = np.array(Image.open(self.masks_dir / fname))
        mask = torch.from_numpy(mask).long()

        caption = str(row[self.caption_col])
        return img, mask, caption

In [8]:
# DataLoader hyperparameters (shared across all ablations)
BATCH_SIZE  = 8
NUM_WORKERS = 4


def make_loaders(caption_col):
    """Build train/val/test loaders for a given caption column."""
    train_dataset = RSMultiModalDataset(train_df, LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    val_dataset   = RSMultiModalDataset(val_df,   LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    test_dataset  = RSMultiModalDataset(test_df,  LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader


# Sanity check with the Phase 2 best caption
train_loader, val_loader, test_loader = make_loaders(BEST_CAPTION)
print(f'Caption column: {BEST_CAPTION}')
print(f'Train: {len(train_loader.dataset)} samples, {len(train_loader)} batches')

imgs, masks, captions = next(iter(train_loader))
print(f'Batch: images {imgs.shape}, masks {masks.shape}')

Caption column: text_qwen3-4b
Train: 7000 samples, 875 batches
Batch: images torch.Size([8, 3, 256, 256]), masks torch.Size([8, 256, 256])


## 3. UNetFormer architecture

Same UNetFormer as in `03_baseline_models.ipynb` and `04_multimodal.ipynb`: a `swsl_resnet18` CNN encoder with a Global-Local Transformer Block decoder, Feature Refinement Head, and auxiliary head for deep supervision. Re-defined here so this notebook is self-contained.

In [9]:
from einops import rearrange
from timm.models.layers import DropPath, trunc_normal_
import timm


class ConvBNReLU(nn.Sequential):
    """Conv2d -> BatchNorm -> ReLU6."""
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels), nn.ReLU6())


class ConvBN(nn.Sequential):
    """Conv2d -> BatchNorm (no activation)."""
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels))


class Conv(nn.Sequential):
    """Standalone Conv2d, no normalization or activation."""
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2))


class SeparableConvBN(nn.Sequential):
    """Depthwise separable convolution: depthwise -> BatchNorm -> pointwise (1x1)."""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, dilation=1,
                 norm_layer=nn.BatchNorm2d):
        super().__init__(
            nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, dilation=dilation,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2,
                      groups=in_channels, bias=False),
            norm_layer(out_channels),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False))

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [10]:
class Mlp(nn.Module):
    """Two-layer MLP using 1x1 convolutions, used inside transformer blocks."""

    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.ReLU6, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Conv2d(in_features, hidden_features, 1, 1, 0, bias=True)
        self.act = act_layer()
        self.fc2 = nn.Conv2d(hidden_features, out_features, 1, 1, 0, bias=True)
        self.drop = nn.Dropout(drop, inplace=True)

    def forward(self, x):
        x = self.fc1(x); x = self.act(x); x = self.drop(x)
        x = self.fc2(x); x = self.drop(x); return x


class GlobalLocalAttention(nn.Module):
    """Window self-attention (global) combined with a local conv branch."""

    def __init__(self, dim=256, num_heads=16, qkv_bias=False,
                 window_size=8, relative_pos_embedding=True):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // self.num_heads
        self.scale = head_dim ** -0.5
        self.ws = window_size
        self.qkv = Conv(dim, 3 * dim, kernel_size=1, bias=qkv_bias)
        self.local1 = ConvBN(dim, dim, kernel_size=3)
        self.local2 = ConvBN(dim, dim, kernel_size=1)
        self.proj = SeparableConvBN(dim, dim, kernel_size=window_size)
        self.attn_x = nn.AvgPool2d(kernel_size=(window_size, 1), stride=1,
                                    padding=(window_size // 2 - 1, 0))
        self.attn_y = nn.AvgPool2d(kernel_size=(1, window_size), stride=1,
                                    padding=(0, window_size // 2 - 1))
        self.relative_pos_embedding = relative_pos_embedding
        if self.relative_pos_embedding:
            self.relative_position_bias_table = nn.Parameter(
                torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads))
            coords_h = torch.arange(self.ws); coords_w = torch.arange(self.ws)
            coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
            coords_flatten = torch.flatten(coords, 1)
            relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
            relative_coords = relative_coords.permute(1, 2, 0).contiguous()
            relative_coords[:, :, 0] += self.ws - 1; relative_coords[:, :, 1] += self.ws - 1
            relative_coords[:, :, 0] *= 2 * self.ws - 1
            relative_position_index = relative_coords.sum(-1)
            self.register_buffer("relative_position_index", relative_position_index)
            trunc_normal_(self.relative_position_bias_table, std=.02)

    def pad(self, x, ps):
        _, _, H, W = x.size()
        if W % ps != 0: x = F.pad(x, (0, ps - W % ps), mode='reflect')
        if H % ps != 0: x = F.pad(x, (0, 0, 0, ps - H % ps), mode='reflect')
        return x

    def pad_out(self, x):
        return F.pad(x, pad=(0, 1, 0, 1), mode='reflect')

    def forward(self, x):
        B, C, H, W = x.shape
        local = self.local2(x) + self.local1(x)
        x = self.pad(x, self.ws); B, C, Hp, Wp = x.shape
        qkv = self.qkv(x)
        q, k, v = rearrange(qkv, 'b (qkv h d) (hh ws1) (ww ws2) -> qkv (b hh ww) h (ws1 ws2) d',
            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
            qkv=3, ws1=self.ws, ws2=self.ws)
        dots = (q @ k.transpose(-2, -1)) * self.scale
        if self.relative_pos_embedding:
            relative_position_bias = self.relative_position_bias_table[
                self.relative_position_index.view(-1)].view(self.ws * self.ws, self.ws * self.ws, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            dots += relative_position_bias.unsqueeze(0)
        attn = dots.softmax(dim=-1); attn = attn @ v
        attn = rearrange(attn, '(b hh ww) h (ws1 ws2) d -> b (h d) (hh ws1) (ww ws2)',
            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
            ws1=self.ws, ws2=self.ws)
        attn = attn[:, :, :H, :W]
        out = self.attn_x(F.pad(attn, pad=(0, 0, 0, 1), mode='reflect')) + \
              self.attn_y(F.pad(attn, pad=(0, 1, 0, 0), mode='reflect'))
        out = out + local; out = self.pad_out(out); out = self.proj(out)
        out = out[:, :, :H, :W]
        return out

In [11]:
class Block(nn.Module):
    """Global-Local Transformer Block (GLTB): pre-norm attention + MLP with residuals."""

    def __init__(self, dim=256, num_heads=16, mlp_ratio=4., qkv_bias=False,
                 drop=0., drop_path=0.,
                 act_layer=nn.ReLU6, norm_layer=nn.BatchNorm2d, window_size=8):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = GlobalLocalAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                                         window_size=window_size)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, out_features=dim,
                       act_layer=act_layer, drop=drop)
        self.norm2 = norm_layer(dim)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

In [12]:
class WF(nn.Module):
    """Weighted feature fusion: upsamples low-resolution features and combines them
    with skip-connection features using learned, ReLU-normalized weights."""

    def __init__(self, in_channels=128, decode_channels=128, eps=1e-8):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = eps
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        return self.post_conv(x)


class FeatureRefinementHead(nn.Module):
    """Final decoder stage: weighted fusion + spatial and channel attention + residual proj."""

    def __init__(self, in_channels=64, decode_channels=64):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = 1e-8
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)
        self.pa = nn.Sequential(
            nn.Conv2d(decode_channels, decode_channels, kernel_size=3, padding=1,
                      groups=decode_channels), nn.Sigmoid())
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            Conv(decode_channels, decode_channels // 16, kernel_size=1), nn.ReLU6(),
            Conv(decode_channels // 16, decode_channels, kernel_size=1), nn.Sigmoid())
        self.shortcut = ConvBN(decode_channels, decode_channels, kernel_size=1)
        self.proj = SeparableConvBN(decode_channels, decode_channels, kernel_size=3)
        self.act = nn.ReLU6()

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        x = self.post_conv(x); shortcut = self.shortcut(x)
        pa = self.pa(x) * x; ca = self.ca(x) * x
        x = pa + ca; x = self.proj(x) + shortcut
        return self.act(x)


class AuxHead(nn.Module):
    """Auxiliary segmentation head used during training for deep supervision."""

    def __init__(self, in_channels=64, num_classes=8):
        super().__init__()
        self.conv = ConvBNReLU(in_channels, in_channels)
        self.drop = nn.Dropout(0.1)
        self.conv_out = Conv(in_channels, num_classes, kernel_size=1)

    def forward(self, x, h, w):
        feat = self.conv(x); feat = self.drop(feat); feat = self.conv_out(feat)
        return F.interpolate(feat, size=(h, w), mode='bilinear', align_corners=False)

## 4. Multimodal architecture

Three additions to the vision-only UNetFormer:

1. **Frozen CLIP text encoder** (`ViT-B-32`, `laion2b_s34b_b79k`): converts each caption into a 512-d L2-normalized embedding. No gradients flow into CLIP.
2. **Text-Visual Cross-Attention**: visual features attend to the text embedding via a gated residual. The gate starts at zero, so text influence ramps up gradually during training instead of disrupting the pretrained backbone.
3. **TextAugmentedDecoder**: same as the UNetFormer decoder, with a cross-attention module after each Global-Local Transformer Block and after the Feature Refinement Head.

Only the cross-attention modules and the decoder are trained. The CNN encoder (`swsl_resnet18`) and CLIP text encoder are both frozen. Identical to `04_multimodal.ipynb`.

In [13]:
import open_clip


class CLIPTextEncoder(nn.Module):
    """Frozen CLIP text encoder. Outputs L2-normalized 512-d embeddings."""

    def __init__(self, model_name='ViT-B-32', pretrained='laion2b_s34b_b79k'):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
        self.clip_model = clip_model
        self.tokenizer = open_clip.get_tokenizer(model_name)

        # Drop the visual tower since only the text encoder is used (saves ~88M frozen params)
        if hasattr(self.clip_model, 'visual'):
            del self.clip_model.visual

        # Freeze all remaining CLIP parameters
        for param in self.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def forward(self, text_list):
        device = next(self.parameters()).device
        tokens = self.tokenizer(text_list).to(device)
        text_features = self.clip_model.encode_text(tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return text_features


# Sanity check
clip_enc = CLIPTextEncoder().to(device)
with torch.no_grad():
    emb = clip_enc(['test caption'])
    print(f'CLIP text encoder loaded: output {emb.shape}')
    print(f'Frozen params: {sum(p.numel() for p in clip_enc.parameters()) / 1e6:.1f}M')
del clip_enc
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIP text encoder loaded: output torch.Size([1, 512])
Frozen params: 63.4M


In [14]:
class TextVisualCrossAttention(nn.Module):
    """Visual features attend to text embedding via gated residual.

    The gate starts at 0, so text initially has no influence. As training
    progresses, the gate learns to weight the text contribution.
    """

    def __init__(self, visual_dim=64, text_dim=512, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = visual_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Project text embedding into the visual feature space
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, visual_dim), nn.ReLU6(),
            nn.Linear(visual_dim, visual_dim),
        )

        self.q_proj = nn.Conv2d(visual_dim, visual_dim, 1)
        self.k_proj = nn.Linear(visual_dim, visual_dim)
        self.v_proj = nn.Linear(visual_dim, visual_dim)
        self.out_proj = nn.Sequential(
            nn.Conv2d(visual_dim, visual_dim, 1), nn.BatchNorm2d(visual_dim),
        )
        # Gated residual — starts at 0, learns to scale text contribution
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, visual, text_embed):
        B, C, H, W = visual.shape
        text_feat = self.text_proj(text_embed)
        q = self.q_proj(visual).reshape(B, self.num_heads, self.head_dim, H * W).permute(0, 1, 3, 2)
        k = self.k_proj(text_feat).reshape(B, self.num_heads, 1, self.head_dim)
        v = self.v_proj(text_feat).reshape(B, self.num_heads, 1, self.head_dim)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        out = self.out_proj(out)
        return visual + self.gate * out

In [15]:
class TextAugmentedDecoder(nn.Module):
    """UNetFormer decoder + cross-attention after each decoder stage."""

    def __init__(self, encoder_channels=(64, 128, 256, 512), decode_channels=64,
                 dropout=0.1, window_size=8, num_classes=7, text_dim=512):
        super().__init__()
        self.pre_conv = ConvBN(encoder_channels[-1], decode_channels, kernel_size=1)
        self.b4 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p3 = WF(encoder_channels[-2], decode_channels)
        self.b3 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p2 = WF(encoder_channels[-3], decode_channels)
        self.b2 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p1 = FeatureRefinementHead(encoder_channels[-4], decode_channels)
        self.up4 = nn.UpsamplingBilinear2d(scale_factor=4)
        self.up3 = nn.UpsamplingBilinear2d(scale_factor=2)
        self.aux_head = AuxHead(decode_channels, num_classes)
        self.segmentation_head = nn.Sequential(
            ConvBNReLU(decode_channels, decode_channels),
            nn.Dropout2d(p=dropout, inplace=True),
            Conv(decode_channels, num_classes, kernel_size=1),
        )

        # Text-visual cross-attention at each decoder stage
        self.text_attn4 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn3 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn2 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn1 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)

        self.init_weight()

    def forward(self, res1, res2, res3, res4, h, w, text_embed):
        if self.training:
            x = self.b4(self.pre_conv(res4)); x = self.text_attn4(x, text_embed); h4 = self.up4(x)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_attn3(x, text_embed); h3 = self.up3(x)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_attn2(x, text_embed); h2 = x
            x = self.p1(x, res1); x = self.text_attn1(x, text_embed)
            x = self.segmentation_head(x)
            x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)
            ah = h4 + h3 + h2; ah = self.aux_head(ah, h, w)
            return x, ah
        else:
            x = self.b4(self.pre_conv(res4)); x = self.text_attn4(x, text_embed)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_attn3(x, text_embed)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_attn2(x, text_embed)
            x = self.p1(x, res1); x = self.text_attn1(x, text_embed)
            x = self.segmentation_head(x)
            return F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)

    def init_weight(self):
        for m in self.children():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

In [16]:
class UNetFormerCLIP(nn.Module):
    """UNetFormer + frozen CLIP text encoder + cross-attention fusion."""

    def __init__(self, decode_channels=64, dropout=0.1, backbone_name='swsl_resnet18',
                 pretrained=True, window_size=8, num_classes=7,
                 clip_model='ViT-B-32', clip_pretrained='laion2b_s34b_b79k'):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, features_only=True, output_stride=32,
                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)
        encoder_channels = self.backbone.feature_info.channels()
        self.text_encoder = CLIPTextEncoder(clip_model, clip_pretrained)
        self.decoder = TextAugmentedDecoder(encoder_channels, decode_channels, dropout,
                                            window_size, num_classes, text_dim=512)

    def forward(self, x, captions):
        h, w = x.size()[-2:]
        res1, res2, res3, res4 = self.backbone(x)
        text_embed = self.text_encoder(captions)
        if self.training:
            out, ah = self.decoder(res1, res2, res3, res4, h, w, text_embed)
            return out, ah
        else:
            return self.decoder(res1, res2, res3, res4, h, w, text_embed)


# Sanity check: instantiate, count parameters, forward pass in both modes
test_model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)

total_params     = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params
print(f'Total parameters    : {total_params / 1e6:.2f}M')
print(f'Trainable parameters: {trainable_params / 1e6:.2f}M')
print(f'Frozen parameters   : {frozen_params / 1e6:.2f}M (CLIP text encoder)')

with torch.no_grad():
    sample_imgs, _, sample_caps = next(iter(train_loader))
    sample_imgs = sample_imgs[:2].to(device)
    sample_caps = list(sample_caps[:2])

    test_model.eval()
    out_eval = test_model(sample_imgs, sample_caps)
    print(f'\nEval  mode: input {sample_imgs.shape} -> output {out_eval.shape}')

    test_model.train()
    out_main, out_aux = test_model(sample_imgs, sample_caps)
    print(f'Train mode: main {out_main.shape}, aux {out_aux.shape}')

del test_model
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Total parameters    : 75.37M
Trainable parameters: 11.94M
Frozen parameters   : 63.43M (CLIP text encoder)


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



Eval  mode: input torch.Size([2, 3, 256, 256]) -> output torch.Size([2, 7, 256, 256])
Train mode: main torch.Size([2, 7, 256, 256]), aux torch.Size([2, 7, 256, 256])


## 5. Training infrastructure

Same training recipe as `04_multimodal.ipynb`: 30 epochs, AdamW (lr=6e-4, weight decay 0.01), CosineAnnealingLR (eta_min=1e-6), weighted CE + 0.4 × auxiliary head loss.

The `train_multimodal` function below accepts optional `model_factory` and `criterion` arguments. Defaults reproduce the Phase 2 configuration (`UNetFormerCLIP` with weighted CE). Sections 6-9 supply different models (RemoteCLIP, FiLM, multi-token variants) and different losses (CE+Dice, CE+Lovász, CE+Tversky) without modifying the training loop.

Each run is logged to Weights & Biases under the project `di725_project`.

In [17]:
# Training hyperparameters (same as 04_multimodal.ipynb)
NUM_EPOCHS      = 30
LR              = 6e-4
WEIGHT_DECAY    = 0.01
AUX_LOSS_WEIGHT = 0.4

In [18]:
def update_confusion(intersection, union, pred, target, num_classes=NUM_CLASSES):
    """Accumulate per-class intersection and union counts in-place.
    pred, target: tensors of shape (B, H, W) with class indices.
    Computation stays on GPU; only scalars transfer to CPU via .item()."""
    for c in range(num_classes):
        pred_c   = (pred == c)
        target_c = (target == c)
        intersection[c] += (pred_c & target_c).sum().item()
        union[c]        += (pred_c | target_c).sum().item()


def finalize_iou(intersection, union, num_classes=NUM_CLASSES):
    """Convert accumulated counts into per-class IoU and mIoU.
    Classes absent from both pred and target (union=0) are NaN and excluded from mIoU."""
    class_ious = []
    for c in range(num_classes):
        if union[c] == 0:
            class_ious.append(float('nan'))
        else:
            class_ious.append(intersection[c] / union[c])
    valid = [x for x in class_ious if not np.isnan(x)]
    miou = np.mean(valid) if valid else 0.0
    return class_ious, miou

In [19]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """One training epoch over the (image, mask, caption) loader.

    The criterion takes (logits, target_mask) and returns a scalar loss.
    Used for both weighted CE and composite losses like CE+Dice, CE+Lovász, CE+Tversky.
    """
    model.train()
    total_loss = 0
    for imgs, masks, captions in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits_main, logits_aux = model(imgs, list(captions))
        loss = criterion(logits_main, masks) + AUX_LOSS_WEIGHT * criterion(logits_aux, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes=NUM_CLASSES):
    """Returns: per-class pixel IoU list, pixel-level mIoU, overall accuracy, average loss."""
    model.eval()
    intersection = np.zeros(num_classes, dtype=np.int64)
    union        = np.zeros(num_classes, dtype=np.int64)
    total_correct = 0
    total_pixels  = 0
    total_loss    = 0
    for imgs, masks, captions in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs, list(captions))
        total_loss += criterion(logits, masks).item()
        preds = logits.argmax(dim=1)
        update_confusion(intersection, union, preds, masks, num_classes)
        total_correct += (preds == masks).sum().item()
        total_pixels  += masks.numel()
    class_ious, miou = finalize_iou(intersection, union, num_classes)
    oa = total_correct / total_pixels
    return class_ious, miou, oa, total_loss / len(loader)

In [20]:
def train_multimodal(caption_col, model_factory=None, criterion=None,
                     num_epochs=NUM_EPOCHS, lr=LR, run_name=None, wandb_config=None):
    """Train a multimodal model and return (history, best_val_miou, checkpoint_path).

    Args:
        caption_col:   caption column from CAPTION_COLS to use as text input.
        model_factory: callable returning a fresh model.
                       Defaults to the Phase 2 model (CLIP + pooled cross-attention).
        criterion:     loss function (logits, target) -> scalar. Defaults to weighted CE.
        run_name:      identifier for the W&B run and checkpoint filename.
                       Defaults to f'multimodal_{caption_col}'.
        wandb_config:  dict added to the W&B run config. Augments the default config
                       with experiment-specific fields (loss type, fusion type, encoder).
    """
    # Defaults reproduce the Phase 2 configuration
    if model_factory is None:
        model_factory = lambda: UNetFormerCLIP(num_classes=NUM_CLASSES)
    if criterion is None:
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    if run_name is None:
        base_name = caption_col.replace('-', '_').replace('/', '_')
        run_name = f'multimodal_{base_name}'

    save_path = CHECKPOINTS_DIR / f'{run_name}_best.pth'

    # Loaders and model
    train_loader, val_loader, _ = make_loaders(caption_col)
    model = model_factory().to(device)

    # Optimizer over only trainable params (skips frozen CLIP)
    trainable = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    # W&B run
    default_config = {
        'caption_col':     caption_col,
        'num_epochs':      num_epochs,
        'lr':              lr,
        'weight_decay':    WEIGHT_DECAY,
        'batch_size':      BATCH_SIZE,
        'aux_loss_weight': AUX_LOSS_WEIGHT,
        'seed':            SEED,
    }
    if wandb_config:
        default_config.update(wandb_config)

    wandb.init(project=WANDB_PROJECT, name=run_name, config=default_config, reinit=True)

    history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_oa': []}
    best_miou = 0.0

    print(f'Training {run_name} for {num_epochs} epochs...')
    print(f"{'Epoch':>5} {'T_Loss':>8} {'V_Loss':>8} {'V_mIoU':>8} {'V_OA':>8} {'Time':>7} {'':>6}")
    print('-' * 55)

    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_class_ious, val_miou, val_oa, val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_miou'].append(val_miou)
        history['val_oa'].append(val_oa)

        log_dict = {
            'epoch':      epoch + 1,
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'val_miou':   val_miou,
            'val_oa':     val_oa,
            'lr':         scheduler.get_last_lr()[0],
        }
        for c, name in enumerate(CLASS_NAMES):
            if not np.isnan(val_class_ious[c]):
                log_dict[f'val_iou/{name}'] = val_class_ious[c]
        wandb.log(log_dict)

        status = ''
        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(model.state_dict(), str(save_path))
            status = 'BEST'

        elapsed = time.time() - t0
        print(f'{epoch+1:>5} {train_loss:>8.4f} {val_loss:>8.4f} {val_miou:>8.4f} {val_oa:>8.4f} {elapsed:>6.1f}s {status:>6}')

    wandb.summary['best_val_miou'] = best_miou
    wandb.finish()

    # Free the training model from GPU memory before the next experiment
    del model
    torch.cuda.empty_cache()

    print(f'\nBest validation mIoU: {best_miou:.4f}')
    return history, best_miou, save_path

In [21]:
def save_history(history, name):
    """Persist training history to disk so it survives runtime restarts."""
    path = RESULTS_DIR / f'{name}_history.json'
    with open(path, 'w') as f:
        json.dump(history, f)
    print(f'Saved history to {path}')


def evaluate_checkpoint(caption_col, ckpt_path, model_factory=None, criterion=None):
    """Reload a trained checkpoint and evaluate on the test set."""
    if model_factory is None:
        model_factory = lambda: UNetFormerCLIP(num_classes=NUM_CLASSES)
    if criterion is None:
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    _, _, test_loader = make_loaders(caption_col)
    model = model_factory().to(device)
    model.load_state_dict(torch.load(str(ckpt_path)))
    class_ious, miou, oa, loss = evaluate(model, test_loader, criterion, device)
    del model
    torch.cuda.empty_cache()
    return class_ious, miou, oa, loss

## 6. Ablation 1 — Text encoder pretraining

Tests whether replacing CLIP with RemoteCLIP changes segmentation performance. CLIP (`ViT-B-32`, `laion2b_s34b_b79k`) is pretrained on generic web image-text pairs. RemoteCLIP (Liu et al., 2023) is a CLIP variant fine-tuned on remote-sensing image-text datasets (RSITMD, RSICD, Sydney-Captions). Both share the same `ViT-B-32` architecture, so only the encoder weights change.

All 5 caption variants from Phase 2 are evaluated to test whether the encoder swap helps uniformly or whether the benefit depends on caption type. Phase 2 CLIP results are loaded from `multimodal_results.json` as the reference.

Reference: Liu et al., "RemoteCLIP: A Vision Language Foundation Model for Remote Sensing." IEEE TGRS 2023.

In [22]:
from huggingface_hub import hf_hub_download


class CLIPTextEncoderRC(nn.Module):
    """Frozen text encoder that supports either CLIP or RemoteCLIP weights.

    Both use `ViT-B-32` architecture; only the trained weights differ. RemoteCLIP
    weights are downloaded from HuggingFace (chendelong/RemoteCLIP) and loaded
    into an open_clip ViT-B-32 model with no pretrained weights.
    """

    def __init__(self, model_name='ViT-B-32', pretrained='laion2b_s34b_b79k'):
        super().__init__()
        if pretrained == 'remoteclip':
            # Create an empty ViT-B-32, then load RemoteCLIP weights into it
            clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=None)
            ckpt_path = hf_hub_download(
                'chendelong/RemoteCLIP',
                f'RemoteCLIP-{model_name}.pt',
                cache_dir=str(REMOTECLIP_CACHE),
            )
            ckpt = torch.load(ckpt_path, map_location='cpu')
            msg = clip_model.load_state_dict(ckpt)
            print(f'RemoteCLIP {model_name} loaded: {msg}')
        else:
            clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)

        self.clip_model = clip_model
        self.tokenizer = open_clip.get_tokenizer(model_name)

        if hasattr(self.clip_model, 'visual'):
            del self.clip_model.visual

        for param in self.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def forward(self, text_list):
        device = next(self.parameters()).device
        tokens = self.tokenizer(text_list).to(device)
        text_features = self.clip_model.encode_text(tokens)
        return text_features / text_features.norm(dim=-1, keepdim=True)


class UNetFormerRemoteCLIP(nn.Module):
    """UNetFormer + frozen RemoteCLIP text encoder + cross-attention fusion.

    Architecture is identical to UNetFormerCLIP except for the text encoder weights.
    """

    def __init__(self, decode_channels=64, dropout=0.1, backbone_name='swsl_resnet18',
                 pretrained=True, window_size=8, num_classes=7):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, features_only=True, output_stride=32,
                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)
        encoder_channels = self.backbone.feature_info.channels()
        self.text_encoder = CLIPTextEncoderRC(pretrained='remoteclip')
        self.decoder = TextAugmentedDecoder(encoder_channels, decode_channels, dropout,
                                            window_size, num_classes, text_dim=512)

    def forward(self, x, captions):
        h, w = x.size()[-2:]
        res1, res2, res3, res4 = self.backbone(x)
        text_embed = self.text_encoder(captions)
        if self.training:
            out, ah = self.decoder(res1, res2, res3, res4, h, w, text_embed)
            return out, ah
        else:
            return self.decoder(res1, res2, res3, res4, h, w, text_embed)


# Sanity check
test_model = UNetFormerRemoteCLIP(num_classes=NUM_CLASSES).to(device)
total = sum(p.numel() for p in test_model.parameters()) / 1e6
trainable = sum(p.numel() for p in test_model.parameters() if p.requires_grad) / 1e6
print(f'Total parameters    : {total:.2f}M')
print(f'Trainable parameters: {trainable:.2f}M (same as CLIP-based model)')

with torch.no_grad():
    sample_imgs, _, sample_caps = next(iter(train_loader))
    sample_imgs = sample_imgs[:2].to(device)
    sample_caps = list(sample_caps[:2])
    test_model.eval()
    out = test_model(sample_imgs, sample_caps)
    print(f'Forward pass: {sample_imgs.shape} -> {out.shape}')

del test_model
torch.cuda.empty_cache()

RemoteCLIP-ViT-B-32.pt:   0%|          | 0.00/605M [00:00<?, ?B/s]

RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>
Total parameters    : 75.37M
Trainable parameters: 11.94M (same as CLIP-based model)
Forward pass: torch.Size([2, 3, 256, 256]) -> torch.Size([2, 7, 256, 256])


In [24]:
# Train UNetFormerRemoteCLIP on all 5 caption variants
remoteclip_factory = lambda: UNetFormerRemoteCLIP(num_classes=NUM_CLASSES)
remoteclip_runs = {}

for caption_col in CAPTION_COLS:
    base_name = caption_col.replace('-', '_').replace('/', '_')
    run_name = f'remoteclip_{base_name}'

    print(f'\n{"=" * 60}')
    print(f'Caption: {caption_col}')
    print(f'{"=" * 60}\n')

    hist, val_miou, ckpt = train_multimodal(
        caption_col=caption_col,
        model_factory=remoteclip_factory,
        run_name=run_name,
        wandb_config={'text_encoder': 'remoteclip', 'experiment': 'ablation_1_encoder'},
    )
    save_history(hist, run_name)
    remoteclip_runs[caption_col] = {'history': hist, 'val_miou': val_miou, 'ckpt': ckpt}


Caption: hybrid_gemma3-4b



RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Training remoteclip_hybrid_gemma3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.0835   0.4844   0.5050   0.7364   60.5s   BEST
    2   0.7133   0.4117   0.4973   0.7657   58.1s       
    3   0.6179   0.3728   0.5117   0.7669   60.0s   BEST
    4   0.5877   0.5599   0.4153   0.6985   59.0s       
    5   0.5726   0.4775   0.4619   0.6987   58.6s       
    6   0.5288   0.3129   0.5646   0.8180   58.4s   BEST
    7   0.4797   0.3084   0.6055   0.8248   58.3s   BEST
    8   0.4805   0.3095   0.5750   0.8082   56.5s       
    9   0.4450   0.2879   0.5863   0.8156   56.9s       
   10   0.4300   0.2829   0.5905   0.8312   56.5s       
   11   0.4060   0.2659   0.6327   0.8356   57.5s   BEST
   12   0.3950   0.3036   0.6086   0.8186   57.5s       
   13   0.3924   0.2577   0.6349   0.8448   57.6s   BEST
   14   0.3726   0.2560   0.6536   0.8582   60.2s   BEST
   15   0.3500   0.2755   0.6368   

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▃▅▃▄▁▅▄▄▄▇▅▅▇▆▆▆▄█▅▆▇▇████████
val_iou/Built-up,▂▄▃▃▁▅▇▄▆▅█▇▆▇█▆█▆▆▇▇██▇██▇█▇█
val_iou/Crop,▁▂▃▂▂▄▃▅▄▅▅▃▆▇▃▅▅▆▅▇▇█████████
val_iou/Grass,▁▃▂▃▁▅▅▅▅▆▆▅▆▇▆▇▆▇▆▇▇▇████████
val_iou/Shrub,▁▃▃▃▁▄▅▃▃▃▄▃▃▅▆▄▆▄▅▇▇▇▇▇▇█████
val_iou/Tree,▅▅▆▁▄▇▇▇▇▇▇▇▇▇▇▇▇█▇███████████
val_iou/Water,▇▄▆▁▆▆▇█▇▆████████▇███████████
+3,...


...


Best validation mIoU: 0.6466
Saved history to /content/drive/MyDrive/DI725_Project/results/remoteclip_vision_gemma3_4b_history.json

Caption: vision_qwen3-vl-8b



RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


Training remoteclip_vision_qwen3_vl_8b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.3048   0.9714   0.3889   0.5408   60.3s   BEST
    2   0.9791   0.6384   0.4579   0.7237   59.7s   BEST
    3   0.9060   0.6255   0.4144   0.5975   58.3s       
    4   0.8083   0.5572   0.4748   0.6325   58.3s   BEST
    5   0.7844   0.4579   0.5090   0.7347   58.9s   BEST
    6   0.7433   0.5209   0.5395   0.7499   59.9s   BEST
    7   0.7111   0.4327   0.5392   0.7547   58.8s       
    8   0.6752   0.4693   0.4857   0.6778   59.4s       
    9   0.6347   0.4390   0.5187   0.7326   59.3s       
   10   0.6127   0.4162   0.5157   0.7367   59.5s       
   11   0.6003   0.4152   0.5151   0.7515   58.9s       
   12   0.5781   0.4570   0.5142   0.7216   58.8s       
   13   0.5470   0.4107   0.5830   0.8093   59.1s   BEST
   14   0.5199   0.3704   0.5816   0.8034   57.8s       
   15   0.5049   0.3917   0.5833 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▆▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▂▁▁▃▆▅▂▄▆▄▁▇▅▆▅▆▇▆▅▆▇███▇▇▇▇▇
val_iou/Built-up,▁▂▁▆▇█▆▄▄▆▄▄▆▆▇▇██▇▇▇███▇▇▇▇▇█
val_iou/Crop,▁▄▆▆▅▆▆▆▇▆▆▇▇▇▆▇▇█▇▇██████████
val_iou/Grass,▁▅▄▄▅▅▆▅▆▅▆▅▇▇▇▇▇█▇▇▇█████████
val_iou/Shrub,▁▄▁▁▄▃▃▂▂▃▃▃▅▅▅▅▆▅▅▆▆▇██▇█████
val_iou/Tree,▅▆▁▁▆▆▅▃▅▆▆▅▆▇▇▆▇▇▇▆▇█████████
val_iou/Water,▅▁▃▆▂▄▆▆▆▂▄█▆█▇▅█▇▆▇▇███▇█████
+3,...



Best validation mIoU: 0.6458
Saved history to /content/drive/MyDrive/DI725_Project/results/remoteclip_vision_qwen3_vl_8b_history.json


In [48]:
# Test set evaluation for all 5 RemoteCLIP checkpoints
import contextlib, io

remoteclip_test = {}

print('Evaluating RemoteCLIP on 5 caption variants...\n')
print(f"{'Caption':<22} {'test mIoU':>11} {'test OA':>10}")
print('-' * 45)

for caption_col, run_info in remoteclip_runs.items():
    # Swallow the "RemoteCLIP loaded:" prints from CLIPTextEncoderRC.__init__
    # so the comparison table stays clean. The load itself still happens.
    with contextlib.redirect_stdout(io.StringIO()):
        class_ious, miou, oa, loss = evaluate_checkpoint(
            caption_col=caption_col,
            ckpt_path=run_info['ckpt'],
            model_factory=remoteclip_factory,
        )
    remoteclip_test[caption_col] = {
        'class_ious': class_ious, 'miou': miou, 'oa': oa, 'loss': loss,
    }
    print(f'{caption_col:<22} {miou:>11.4f} {oa:>10.4f}')

Evaluating RemoteCLIP on 5 caption variants...

Caption                  test mIoU    test OA
---------------------------------------------
hybrid_gemma3-4b            0.6961     0.8882
hybrid_qwen3-vl-8b          0.6955     0.8877
text_qwen3-4b               0.7002     0.8923
vision_gemma3-4b            0.6417     0.8530
vision_qwen3-vl-8b          0.6412     0.8546


In [26]:
# Paired comparison: Phase 2 CLIP vs Phase 3 RemoteCLIP, per caption
phase2_clip = multimodal_data['multimodal']

print(f"{'Caption':<22} {'CLIP':>10} {'RemoteCLIP':>12} {'Δ':>10} {'Rel %':>8}")
print('-' * 64)

for caption_col in CAPTION_COLS:
    clip_miou = phase2_clip[caption_col]['test_miou']
    rc_miou   = remoteclip_test[caption_col]['miou']
    delta = rc_miou - clip_miou
    rel = (delta / clip_miou) * 100
    print(f'{caption_col:<22} {clip_miou:>10.4f} {rc_miou:>12.4f} {delta:>+10.4f} {rel:>+7.2f}%')

# Best RemoteCLIP caption (used to select the final-model configuration in Section 9)
best_rc_caption = max(remoteclip_test, key=lambda k: remoteclip_test[k]['miou'])
best_rc_miou = remoteclip_test[best_rc_caption]['miou']
print(f'\nBest RemoteCLIP caption: {best_rc_caption} (test mIoU = {best_rc_miou:.4f})')
print(f'Phase 2 best CLIP caption: {BEST_CAPTION} (test mIoU = {MULTIMODAL_MIOU:.4f})')

Caption                      CLIP   RemoteCLIP          Δ    Rel %
----------------------------------------------------------------
hybrid_gemma3-4b           0.6930       0.6961    +0.0031   +0.45%
hybrid_qwen3-vl-8b         0.6906       0.6955    +0.0050   +0.72%
text_qwen3-4b              0.6970       0.7002    +0.0032   +0.46%
vision_gemma3-4b           0.6393       0.6417    +0.0023   +0.36%
vision_qwen3-vl-8b         0.6519       0.6412    -0.0107   -1.64%

Best RemoteCLIP caption: text_qwen3-4b (test mIoU = 0.7002)
Phase 2 best CLIP caption: text_qwen3-4b (test mIoU = 0.6970)


In [27]:
# Per-class IoU comparison on the best RemoteCLIP caption
best_rc_ious = remoteclip_test[best_rc_caption]['class_ious']
clip_on_best_rc_caption_ious = phase2_clip[best_rc_caption]['class_ious']
clip_on_best_rc_caption_miou = phase2_clip[best_rc_caption]['test_miou']

print(f'Per-class comparison on {best_rc_caption}:\n')
print(f"{'Class':<12} {'CLIP':>10} {'RemoteCLIP':>12} {'Δ':>10} {'Rel %':>8}")
print('-' * 54)

for i, name in enumerate(CLASS_NAMES):
    c = clip_on_best_rc_caption_ious[i]
    r = best_rc_ious[i]
    if c is None or (isinstance(r, float) and np.isnan(r)):
        print(f'{name:<12} {"N/A":>10} {"N/A":>12} {"":>10} {"":>8}')
        continue
    delta = r - c
    rel = (delta / c) * 100 if c > 0 else float('nan')
    print(f'{name:<12} {c:>10.4f} {r:>12.4f} {delta:>+10.4f} {rel:>+7.1f}%')

print('-' * 54)
print(f'{"mIoU":<12} {clip_on_best_rc_caption_miou:>10.4f} {best_rc_miou:>12.4f} '
      f'{best_rc_miou - clip_on_best_rc_caption_miou:>+10.4f} '
      f'{(best_rc_miou - clip_on_best_rc_caption_miou) / clip_on_best_rc_caption_miou * 100:>+7.1f}%')

Per-class comparison on text_qwen3-4b:

Class              CLIP   RemoteCLIP          Δ    Rel %
------------------------------------------------------
Tree             0.8758       0.8761    +0.0002    +0.0%
Shrub            0.3261       0.3401    +0.0139    +4.3%
Grass            0.8057       0.8092    +0.0035    +0.4%
Crop             0.8159       0.8190    +0.0030    +0.4%
Built-up         0.5893       0.5857    -0.0036    -0.6%
Barren           0.5665       0.5869    +0.0204    +3.6%
Water            0.8994       0.8845    -0.0149    -1.7%
------------------------------------------------------
mIoU             0.6970       0.7002    +0.0032    +0.5%


In [28]:
# Persist Ablation 1 results
ablation1_results = {
    'experiment': 'text_encoder_pretraining',
    'phase2_clip_test': {
        cap: {
            'miou':       phase2_clip[cap]['test_miou'],
            'oa':         phase2_clip[cap]['test_oa'],
            'class_ious': phase2_clip[cap]['class_ious'],
        }
        for cap in CAPTION_COLS
    },
    'phase3_remoteclip_test': {
        cap: {
            'miou':       res['miou'],
            'oa':         res['oa'],
            'class_ious': [float(x) if not np.isnan(x) else None for x in res['class_ious']],
        }
        for cap, res in remoteclip_test.items()
    },
    'best_remoteclip_caption': best_rc_caption,
}

with open(RESULTS_DIR / 'ablation1_text_encoder.json', 'w') as f:
    json.dump(ablation1_results, f, indent=2)
print(f"Saved to {RESULTS_DIR / 'ablation1_text_encoder.json'}")

Saved to /content/drive/MyDrive/DI725_Project/results/ablation1_text_encoder.json


**Observations.**

RemoteCLIP improves test mIoU on four of the five caption variants, with the gain concentrated on text-heavy and hybrid captions. The largest improvements are on `hybrid_qwen3-vl-8b` (+0.72%) and `text_qwen3-4b` (+0.46%). The vision-only `vision_gemma3-4b` improves slightly (+0.36%), while `vision_qwen3-vl-8b` regresses (-1.64%), consistent with the Phase 2 finding that vision-only captions are noisier and benefit less from any encoder change.

The best RemoteCLIP caption is `text_qwen3-4b` (test mIoU 0.7002), matching the Phase 2 best CLIP caption (0.6970). The encoder swap preserves the caption ranking but slightly increases the headline number by +0.0032 mIoU (+0.46%).

Per-class on `text_qwen3-4b`, the gains concentrate on rare classes: Shrub +4.3% and Barren +3.6%. Dominant classes (Tree, Grass, Crop) are essentially unchanged, and Built-up (-0.6%) and Water (-1.7%) regress slightly. This pattern is consistent with how domain-adapted text encoders help: RemoteCLIP's remote-sensing pretraining encodes class semantics for rare landscape types more accurately than generic CLIP, which the cross-attention can then exploit. Dominant classes already segment well from visual features alone, so encoder improvements have less room to help there.

The overall improvement is modest (+0.46% mIoU). RemoteCLIP is a useful but not dramatic encoder swap on this dataset and caption distribution.

## 7. Ablation 2 — Loss function

Tests whether region-based losses outperform per-pixel weighted cross-entropy. Three composite losses are compared against the Phase 2 weighted CE. Each pairs weighted CE with a region-based term using `α = 0.5`:

- **CE + Dice** — combines weighted CE with the Dice loss (Milletari et al., 2016), a soft surrogate for the F1 score. Encourages high overlap between predicted and ground-truth regions.
- **CE + Lovász** — combines weighted CE with the Lovász-Softmax loss (Berman et al., 2018), a piecewise-linear surrogate for the Jaccard index (IoU). Directly optimizes the evaluation metric used for mIoU.
- **CE + Tversky** — combines weighted CE with the Tversky loss (Salehi et al., 2017), a generalization of Dice with asymmetric weighting (`α=0.3`, `β=0.7`). The `β > α` setting penalizes false negatives more, biasing toward recall — useful for the rare classes in this dataset.

Pure region-based losses are unstable early in training because their gradients are sparse. Combining with CE keeps a dense gradient signal throughout.

Only the loss changes. Encoder (CLIP), fusion (pooled cross-attention), and caption (`text_qwen3-4b`) match Phase 2.

In [29]:
class DiceLoss(nn.Module):
    """Multi-class Dice loss.

    Computes 1 - mean per-class Dice coefficient across all classes present
    in the batch. Classes absent from the ground truth are excluded so they
    don't dilute the loss.
    """

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, target):
        """logits: (B, C, H, W), target: (B, H, W) with class indices."""
        probs = F.softmax(logits, dim=1)
        target_1h = F.one_hot(target, num_classes=probs.size(1)).permute(0, 3, 1, 2).float()

        # Per-class intersection and cardinality, summed over batch and spatial dims
        dims = (0, 2, 3)
        intersection = (probs * target_1h).sum(dim=dims)
        cardinality  = probs.sum(dim=dims) + target_1h.sum(dim=dims)

        # Only average over classes present in the ground truth
        present = target_1h.sum(dim=dims) > 0
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice[present].mean()


class TverskyLoss(nn.Module):
    """Multi-class Tversky loss (Salehi et al., 2017).

    Generalizes Dice with asymmetric weighting of false positives and false negatives:
        Tversky = TP / (TP + alpha * FP + beta * FN)
    Setting alpha = beta = 0.5 recovers Dice. Setting beta > alpha biases the loss
    toward recall (penalizes false negatives more), which helps rare-class detection.
    """

    def __init__(self, alpha=0.3, beta=0.7, smooth=1.0):
        super().__init__()
        self.alpha  = alpha
        self.beta   = beta
        self.smooth = smooth

    def forward(self, logits, target):
        """logits: (B, C, H, W), target: (B, H, W) with class indices."""
        probs = F.softmax(logits, dim=1)
        target_1h = F.one_hot(target, num_classes=probs.size(1)).permute(0, 3, 1, 2).float()

        dims = (0, 2, 3)
        tp = (probs * target_1h).sum(dim=dims)
        fp = (probs * (1 - target_1h)).sum(dim=dims)
        fn = ((1 - probs) * target_1h).sum(dim=dims)

        present = target_1h.sum(dim=dims) > 0
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return 1.0 - tversky[present].mean()

In [30]:
# Lovász-Softmax (Berman et al., CVPR 2018). Reference implementation adapted to
# multi-class case. Computes a smooth surrogate for the Jaccard (IoU) loss by
# sorting per-class errors and applying the Lovász extension.

def _lovasz_grad(gt_sorted):
    """Gradient of the Lovász extension of the Jaccard loss for sorted ground truth."""
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def _lovasz_softmax_flat(probs, labels, classes='present'):
    """Multi-class Lovász-Softmax on flattened predictions.
    probs:  (P, C) class probabilities
    labels: (P,)   ground-truth labels
    classes: 'present' uses only classes present in labels, 'all' uses every class.
    """
    if probs.numel() == 0:
        return probs * 0.0
    C = probs.size(1)
    losses = []
    class_to_sum = list(range(C)) if classes == 'all' else torch.unique(labels).tolist()
    for c in class_to_sum:
        fg = (labels == c).float()
        if fg.sum() == 0:
            continue
        class_pred = probs[:, c]
        errors = (fg - class_pred).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        fg_sorted = fg[perm]
        losses.append(torch.dot(errors_sorted, _lovasz_grad(fg_sorted)))
    if len(losses) == 0:
        return probs.sum() * 0.0
    return torch.stack(losses).mean()


class LovaszSoftmaxLoss(nn.Module):
    """Multi-class Lovász-Softmax loss for segmentation logits."""

    def __init__(self, classes='present'):
        super().__init__()
        self.classes = classes

    def forward(self, logits, target):
        """logits: (B, C, H, W), target: (B, H, W)."""
        probs = F.softmax(logits, dim=1)
        B, C, H, W = probs.shape
        probs_flat  = probs.permute(0, 2, 3, 1).reshape(-1, C)
        target_flat = target.reshape(-1)
        return _lovasz_softmax_flat(probs_flat, target_flat, classes=self.classes)

In [31]:
class CompositeLoss(nn.Module):
    """Weighted sum of weighted CE and a region-based loss:
        total = alpha * CE(weighted) + (1 - alpha) * region_loss

    Used for CE+Dice, CE+Lovász, and CE+Tversky in this notebook.
    """

    def __init__(self, ce_weight, region_loss, alpha=0.5):
        super().__init__()
        self.alpha       = alpha
        self.ce          = nn.CrossEntropyLoss(weight=ce_weight)
        self.region_loss = region_loss

    def forward(self, logits, target):
        return self.alpha * self.ce(logits, target) + (1.0 - self.alpha) * self.region_loss(logits, target)


# Sanity check on a random batch
def _sanity_loss(name, loss_fn):
    sl = torch.randn(2, NUM_CLASSES, 64, 64, device=device, requires_grad=True)
    st = torch.randint(0, NUM_CLASSES, (2, 64, 64), device=device)
    val = loss_fn(sl, st)
    val.backward()
    ok = sl.grad is not None and torch.isfinite(sl.grad).all().item()
    print(f'{name:<12} value: {val.item():.4f}, gradient OK: {ok}')

_sanity_loss('CE+Dice',    CompositeLoss(class_weights_tensor, DiceLoss()).to(device))
_sanity_loss('CE+Lovász',  CompositeLoss(class_weights_tensor, LovaszSoftmaxLoss()).to(device))
_sanity_loss('CE+Tversky', CompositeLoss(class_weights_tensor, TverskyLoss(alpha=0.3, beta=0.7)).to(device))

CE+Dice      value: 1.5966, gradient OK: True
CE+Lovász    value: 1.5987, gradient OK: True
CE+Tversky   value: 1.6119, gradient OK: True


In [32]:
# Train all three composite losses on the Phase 2 best caption
DICE_ALPHA    = 0.5
LOVASZ_ALPHA  = 0.5
TVERSKY_ALPHA = 0.5
TVERSKY_TV_ALPHA, TVERSKY_TV_BETA = 0.3, 0.7

dice_criterion    = CompositeLoss(class_weights_tensor, DiceLoss(),
                                  alpha=DICE_ALPHA)
lovasz_criterion  = CompositeLoss(class_weights_tensor, LovaszSoftmaxLoss(),
                                  alpha=LOVASZ_ALPHA)
tversky_criterion = CompositeLoss(class_weights_tensor,
                                  TverskyLoss(alpha=TVERSKY_TV_ALPHA, beta=TVERSKY_TV_BETA),
                                  alpha=TVERSKY_ALPHA)

loss_runs = {}

for label, criterion, wandb_extra in [
    ('ce_dice',    dice_criterion,    {'loss_type': 'ce_dice',    'composite_alpha': DICE_ALPHA}),
    ('ce_lovasz',  lovasz_criterion,  {'loss_type': 'ce_lovasz',  'composite_alpha': LOVASZ_ALPHA}),
    ('ce_tversky', tversky_criterion, {'loss_type': 'ce_tversky', 'composite_alpha': TVERSKY_ALPHA,
                                       'tversky_alpha': TVERSKY_TV_ALPHA, 'tversky_beta': TVERSKY_TV_BETA}),
]:
    run_name = f'loss_{label}_text_qwen3_4b'
    print(f'\n{"=" * 60}')
    print(f'Loss: {label}')
    print(f'{"=" * 60}\n')

    hist, val_miou, ckpt = train_multimodal(
        caption_col=BEST_CAPTION,
        criterion=criterion,
        run_name=run_name,
        wandb_config={'experiment': 'ablation_2_loss', **wandb_extra},
    )
    save_history(hist, run_name)
    loss_runs[label] = {'history': hist, 'val_miou': val_miou, 'ckpt': ckpt,
                        'criterion': criterion}


Loss: ce_dice



Training loss_ce_dice_text_qwen3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   0.9646   0.5221   0.4922   0.7457   60.1s   BEST
    2   0.7246   0.4517   0.5211   0.7676   60.0s   BEST
    3   0.6597   0.4495   0.5931   0.8235   59.7s   BEST
    4   0.6260   0.4228   0.5773   0.8105   60.9s       
    5   0.5957   0.3589   0.6282   0.8362   60.5s   BEST
    6   0.5781   0.4026   0.5847   0.8137   59.5s       
    7   0.5492   0.3826   0.6254   0.8299   60.3s       
    8   0.5303   0.3610   0.6274   0.8353   60.6s       
    9   0.5206   0.3382   0.6311   0.8428   61.1s   BEST
   10   0.4968   0.3234   0.6511   0.8566   61.2s   BEST
   11   0.4886   0.3155   0.6513   0.8593   61.1s   BEST
   12   0.4735   0.3287   0.6784   0.8642   60.7s   BEST
   13   0.4591   0.3193   0.6567   0.8551   58.8s       
   14   0.4447   0.3082   0.6378   0.8491   60.2s       
   15   0.4410   0.3088   0.6786   0

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▃▄▃▄▄▅▄▄▅▆▄▄▅▇▆▅▇▇▇▇▇▇▇█▇████
val_iou/Built-up,▂▁▅▅▆▄▅▅▄▅▅▇▆▆▇▆▇▆▇█▆▆▇▇██████
val_iou/Crop,▁▄▃▃▅▄▅▄▆▆▆▆▇▆▇▇▇▇▇█▇█████████
val_iou/Grass,▁▂▅▅▅▄▅▆▅▆▆▆▆▆▇▇▇▇▇▇▇▇████████
val_iou/Shrub,▁▁▃▃▃▂▂▃▄▄▄▇▅▃▄▆▅▆▅▄▅▅▇▇▇▇▇▇█▇
val_iou/Tree,▁▁▄▂▄▄▃▄▆▆▆▇▆▆▆▇▇▇▇▇▇▇████████
val_iou/Water,▁▃▄▃▇▄▇▇▇▇▇██▆▇▇██████████████
+3,...



Best validation mIoU: 0.7304
Saved history to /content/drive/MyDrive/DI725_Project/results/loss_ce_dice_text_qwen3_4b_history.json

Loss: ce_lovasz



Training loss_ce_lovasz_text_qwen3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   0.9498   0.5066   0.5351   0.7660   68.3s   BEST
    2   0.7149   0.4946   0.5302   0.7622   67.4s       
    3   0.6792   0.4344   0.5896   0.8167   68.5s   BEST
    4   0.6380   0.4638   0.5453   0.7730   67.5s       
    5   0.6013   0.3844   0.6166   0.8284   68.7s   BEST
    6   0.5854   0.4013   0.6522   0.8470   68.0s   BEST
    7   0.5650   0.3727   0.6412   0.8487   67.3s       
    8   0.5478   0.4024   0.6065   0.8328   67.6s       
    9   0.5372   0.3661   0.6394   0.8389   67.1s       
   10   0.5298   0.4926   0.6177   0.7983   67.8s       
   11   0.5050   0.3363   0.6666   0.8626   68.3s   BEST
   12   0.4941   0.3525   0.6828   0.8713   68.1s   BEST
   13   0.4826   0.3522   0.6440   0.8564   67.0s       
   14   0.4748   0.3442   0.6709   0.8684   66.7s       
   15   0.4613   0.3335   0.6635  

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▄▁▄▂▃▆▆▄▅▄▆▆▆▇▅▇▇▇▇▇▇██▇██████
val_iou/Built-up,▁▂▆▄▄▆▅▃▆▆▆▇▄▆▆▇▇▇▇▆▇██▇██████
val_iou/Crop,▃▁▅▄▅▅▅▅▆▁▇▇▆▇▇▇▇▇▇▇▇██▇██████
val_iou/Grass,▂▂▄▁▅▅▅▅▅▃▆▇▆▆▆▇▇▇▇▇▇█████████
val_iou/Shrub,▁▃▂▂▃▃▄▅▂▅▄▄▅▄▄▅▄▅▅█▆▇▇▇▇▇█▇▇▇
val_iou/Tree,▁▃▄▄▆▆▆▅▆▂▇▇▆▆▇▇▇▇▇█▇█████████
val_iou/Water,▄▄▁▂▇▇▆▅▇▇▇▇▅▇▇▇▇█████████████
+3,...



Best validation mIoU: 0.7420
Saved history to /content/drive/MyDrive/DI725_Project/results/loss_ce_lovasz_text_qwen3_4b_history.json

Loss: ce_tversky



Training loss_ce_tversky_text_qwen3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   0.9354   0.6283   0.4354   0.7160   61.4s   BEST
    2   0.6889   0.4865   0.5356   0.7907   61.9s   BEST
    3   0.6236   0.4165   0.5923   0.8291   61.2s   BEST
    4   0.5884   0.4260   0.6160   0.8182   61.1s   BEST
    5   0.5624   0.4037   0.6109   0.8233   59.8s       
    6   0.5500   0.3824   0.6086   0.8169   60.2s       
    7   0.5097   0.3352   0.6347   0.8376   61.2s   BEST
    8   0.4863   0.3508   0.6428   0.8479   59.9s   BEST
    9   0.4921   0.3162   0.6489   0.8548   61.4s   BEST
   10   0.4642   0.3607   0.6733   0.8601   60.6s   BEST
   11   0.4561   0.3179   0.6405   0.8504   61.2s       
   12   0.4343   0.3045   0.6752   0.8705   61.4s   BEST
   13   0.4263   0.3724   0.6609   0.8449   61.4s       
   14   0.4226   0.2853   0.6614   0.8623   60.9s       
   15   0.4056   0.3363   0.6749 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▂▄▃▃▅▂▇▆▇▄▅▇▄█▅▅▆▆█▇▇▇█▇█▇███
val_iou/Built-up,▅▁▅▆▆▇▇▆▆█▆▆▇▆█▇▇▇█▇█▇▇███████
val_iou/Crop,▁▅▄▅▂▆▆▆▆▆▆▇▅▇▇▇▇▇▇▇██████████
val_iou/Grass,▁▁▃▃▂▃▃▅▅▆▄▆▄▅▄▆▅▆▆▆▆▇▇▇▇█████
val_iou/Shrub,▃▄▄▂▅▁▄▃▃▃▄▆▃▄▂▆▅▆▆▆▆▇█▇██████
val_iou/Tree,▁▅▆▆▇▆▇▆▇▇▇█▆█▇▇██████████████
val_iou/Water,▁▇▆█▇▆▇██▇▇███████████████████
+3,...



Best validation mIoU: 0.7348
Saved history to /content/drive/MyDrive/DI725_Project/results/loss_ce_tversky_text_qwen3_4b_history.json


In [33]:
# Test set evaluation for all three loss variants
loss_test = {}

print('Evaluating three loss variants on text_qwen3-4b...\n')
print(f"{'Loss':<12} {'test mIoU':>11} {'test OA':>10}")
print('-' * 36)

for label, run_info in loss_runs.items():
    class_ious, miou, oa, _ = evaluate_checkpoint(
        caption_col=BEST_CAPTION,
        ckpt_path=run_info['ckpt'],
        criterion=run_info['criterion'],
    )
    loss_test[label] = {'class_ious': class_ious, 'miou': miou, 'oa': oa}
    print(f'{label:<12} {miou:>11.4f} {oa:>10.4f}')

Evaluating three loss variants on text_qwen3-4b...

Loss           test mIoU    test OA
------------------------------------
ce_dice           0.7299     0.9010
ce_lovasz         0.7387     0.9051
ce_tversky        0.7301     0.9006


In [34]:
# Four-way per-class comparison: weighted CE (Phase 2) vs three region-based losses
print(f'Loss function comparison on {BEST_CAPTION}:\n')
print(f"{'Class':<10} {'wCE':>8} {'CE+Dice':>9} {'CE+Lov':>9} {'CE+Tver':>9} "
      f"{'ΔDice':>8} {'ΔLov':>8} {'ΔTver':>8}")
print('-' * 78)

ce_ious = MULTIMODAL_IOUS

for i, name in enumerate(CLASS_NAMES):
    ce  = ce_ious[i]
    dc  = loss_test['ce_dice']['class_ious'][i]
    lv  = loss_test['ce_lovasz']['class_ious'][i]
    tv  = loss_test['ce_tversky']['class_ious'][i]
    if any(v is None or (isinstance(v, float) and np.isnan(v)) for v in [ce, dc, lv, tv]):
        print(f'{name:<10} {"N/A":>8}')
        continue
    print(f'{name:<10} {ce:>8.4f} {dc:>9.4f} {lv:>9.4f} {tv:>9.4f} '
          f'{dc-ce:>+8.4f} {lv-ce:>+8.4f} {tv-ce:>+8.4f}')

print('-' * 78)
miou_dc = loss_test['ce_dice']['miou']
miou_lv = loss_test['ce_lovasz']['miou']
miou_tv = loss_test['ce_tversky']['miou']
print(f'{"mIoU":<10} {MULTIMODAL_MIOU:>8.4f} {miou_dc:>9.4f} {miou_lv:>9.4f} {miou_tv:>9.4f} '
      f'{miou_dc - MULTIMODAL_MIOU:>+8.4f} '
      f'{miou_lv - MULTIMODAL_MIOU:>+8.4f} '
      f'{miou_tv - MULTIMODAL_MIOU:>+8.4f}')

print()
print(f'Relative improvement vs Phase 2 (mIoU = {MULTIMODAL_MIOU:.4f}):')
print(f'  CE + Dice    : {(miou_dc - MULTIMODAL_MIOU) / MULTIMODAL_MIOU * 100:+.2f}%')
print(f'  CE + Lovász  : {(miou_lv - MULTIMODAL_MIOU) / MULTIMODAL_MIOU * 100:+.2f}%')
print(f'  CE + Tversky : {(miou_tv - MULTIMODAL_MIOU) / MULTIMODAL_MIOU * 100:+.2f}%')

# Best loss (used to select the final-model configuration in Section 9)
best_loss = max(loss_test, key=lambda k: loss_test[k]['miou'])
print(f'\nBest loss: {best_loss} (test mIoU = {loss_test[best_loss]["miou"]:.4f})')

Loss function comparison on text_qwen3-4b:

Class           wCE   CE+Dice    CE+Lov   CE+Tver    ΔDice     ΔLov    ΔTver
------------------------------------------------------------------------------
Tree         0.8758    0.8816    0.8869    0.8835  +0.0058  +0.0111  +0.0076
Shrub        0.3261    0.3510    0.3908    0.3769  +0.0249  +0.0647  +0.0507
Grass        0.8057    0.8268    0.8328    0.8229  +0.0211  +0.0271  +0.0172
Crop         0.8159    0.8201    0.8210    0.8195  +0.0042  +0.0051  +0.0035
Built-up     0.5893    0.7062    0.7113    0.6907  +0.1169  +0.1220  +0.1015
Barren       0.5665    0.6066    0.6182    0.5996  +0.0401  +0.0517  +0.0331
Water        0.8994    0.9171    0.9101    0.9175  +0.0177  +0.0108  +0.0182
------------------------------------------------------------------------------
mIoU         0.6970    0.7299    0.7387    0.7301  +0.0330  +0.0418  +0.0331

Relative improvement vs Phase 2 (mIoU = 0.6970):
  CE + Dice    : +4.73%
  CE + Lovász  : +5.99%
  CE + 

In [35]:
# Persist Ablation 2 results
ablation2_results = {
    'experiment':  'loss_function',
    'caption_col': BEST_CAPTION,
    'phase2_wce': {
        'miou': MULTIMODAL_MIOU, 'oa': MULTIMODAL_OA, 'class_ious': MULTIMODAL_IOUS,
    },
    'phase3_ce_dice': {
        'miou':            loss_test['ce_dice']['miou'],
        'oa':              loss_test['ce_dice']['oa'],
        'class_ious':      [float(x) if not np.isnan(x) else None
                            for x in loss_test['ce_dice']['class_ious']],
        'composite_alpha': DICE_ALPHA,
    },
    'phase3_ce_lovasz': {
        'miou':            loss_test['ce_lovasz']['miou'],
        'oa':              loss_test['ce_lovasz']['oa'],
        'class_ious':      [float(x) if not np.isnan(x) else None
                            for x in loss_test['ce_lovasz']['class_ious']],
        'composite_alpha': LOVASZ_ALPHA,
    },
    'phase3_ce_tversky': {
        'miou':            loss_test['ce_tversky']['miou'],
        'oa':              loss_test['ce_tversky']['oa'],
        'class_ious':      [float(x) if not np.isnan(x) else None
                            for x in loss_test['ce_tversky']['class_ious']],
        'composite_alpha': TVERSKY_ALPHA,
        'tversky_alpha':   TVERSKY_TV_ALPHA,
        'tversky_beta':    TVERSKY_TV_BETA,
    },
    'best_loss': best_loss,
}

with open(RESULTS_DIR / 'ablation2_loss.json', 'w') as f:
    json.dump(ablation2_results, f, indent=2)
print(f"Saved to {RESULTS_DIR / 'ablation2_loss.json'}")

Saved to /content/drive/MyDrive/DI725_Project/results/ablation2_loss.json


**Observations.**

All three region-based losses substantially improve test mIoU. CE+Dice reaches 0.7299 (+4.73% relative to Phase 2 weighted CE), CE+Tversky reaches 0.7301 (+4.75%), and CE+Lovász reaches 0.7387 (+5.99%). Every class improves under all three losses with no regression, indicating that region-based losses are uniformly beneficial for this task regardless of the specific region metric.

CE+Lovász wins clearly: +0.9% over Dice and Tversky on mIoU. The IoU-aligned objective produces consistently higher per-class IoUs because the training loss directly targets the evaluation metric. Dice (F1-aligned) and Tversky (recall-biased) both target related but slightly different region overlap quantities, and the gap between them and Lovász reflects the alignment advantage.

Per-class, Built-up shows the largest gain across all three losses (Dice +0.1169, Lovász +0.1220, Tversky +0.1015), reaching +20.7% relative improvement under Lovász (0.5893 → 0.7113). Built-up is a structurally well-defined class with clear region boundaries — exactly where region-based losses pay off most. Barren and Shrub also improve substantially (+5-6% under Lovász), consistent with region-based losses giving more weight to small regions than per-pixel weighted CE does.

Tversky's recall bias (β=0.7) was designed to help rare classes. It does help Shrub (+0.0507 vs Dice's +0.0249), but Lovász without any recall bias still helps Shrub more (+0.0647). The IoU alignment of Lovász dominates the recall bias of Tversky on this dataset.

## 8. Ablation 3 — Fusion mechanism

Tests how the spatial granularity of text-vision fusion affects segmentation. Three mechanisms are compared:

- **FiLM (feature-wise linear modulation)** — the text embedding is mapped to channel-wise scale (γ) and shift (β) vectors applied uniformly across all spatial positions. No spatial selectivity.
- **Pooled cross-attention (Phase 2)** — the text embedding is one token. Each spatial position computes attention weights over this single token.
- **Multi-token cross-attention** — exposes the full CLIP text token sequence (up to 77 tokens). Each spatial position can attend selectively to different tokens.

The three span a spectrum from coarsest to finest spatial control over how text shapes the prediction.

References:
- FiLM: Perez et al., "FiLM: Visual Reasoning with a General Conditioning Layer." AAAI 2018.
- Cross-attention: Vaswani et al., "Attention is All You Need." NeurIPS 2017.

Only the fusion mechanism changes. Encoder (CLIP), loss (weighted CE), and caption (`text_qwen3-4b`) match Phase 2.

In [36]:
class FiLMFusion(nn.Module):
    """Feature-wise linear modulation.

    For each decoder feature x of shape (B, C, H, W), produces:
        out = (1 + gamma) * x + beta
    where gamma, beta in (B, C) are projections of the text embedding.
    Applied uniformly to all spatial positions — no spatial attention.
    """

    def __init__(self, visual_dim=64, text_dim=512):
        super().__init__()
        self.gamma_proj = nn.Sequential(
            nn.Linear(text_dim, visual_dim), nn.ReLU6(),
            nn.Linear(visual_dim, visual_dim),
        )
        self.beta_proj = nn.Sequential(
            nn.Linear(text_dim, visual_dim), nn.ReLU6(),
            nn.Linear(visual_dim, visual_dim),
        )
        # Gated residual matching Phase 2's TextVisualCrossAttention for fair comparison
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, visual, text_embed):
        B, C, H, W = visual.shape
        gamma = self.gamma_proj(text_embed).view(B, C, 1, 1)
        beta  = self.beta_proj(text_embed).view(B, C, 1, 1)
        out = (1.0 + gamma) * visual + beta
        return visual + self.gate * (out - visual)

In [37]:
class MultiTokenCrossAttention(nn.Module):
    """Cross-attention using the full CLIP text token sequence rather than a
    single pooled embedding. Visual positions attend selectively to different
    tokens, giving the finest spatial control over text conditioning.

    Same gated residual structure as the Phase 2 TextVisualCrossAttention, so the
    only difference is the number of text tokens used as keys/values.
    """

    def __init__(self, visual_dim=64, text_dim=512, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = visual_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Project text tokens (each is text_dim-d) into visual_dim space
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, visual_dim), nn.ReLU6(),
            nn.Linear(visual_dim, visual_dim),
        )

        self.q_proj = nn.Conv2d(visual_dim, visual_dim, 1)
        self.k_proj = nn.Linear(visual_dim, visual_dim)
        self.v_proj = nn.Linear(visual_dim, visual_dim)
        self.out_proj = nn.Sequential(
            nn.Conv2d(visual_dim, visual_dim, 1), nn.BatchNorm2d(visual_dim),
        )
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, visual, text_seq):
        """visual: (B, C, H, W), text_seq: (B, T, text_dim)."""
        B, C, H, W = visual.shape
        T = text_seq.size(1)

        text_feat = self.text_proj(text_seq)  # (B, T, C)

        q = self.q_proj(visual).reshape(B, self.num_heads, self.head_dim, H * W).permute(0, 1, 3, 2)
        k = self.k_proj(text_feat).reshape(B, T, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        v = self.v_proj(text_feat).reshape(B, T, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, heads, H*W, T)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        out = self.out_proj(out)
        return visual + self.gate * out

In [38]:
class CLIPSequenceTextEncoder(nn.Module):
    """Frozen CLIP text encoder that returns the full token sequence (B, T, 512)
    rather than a single pooled embedding. Used by MultiTokenCrossAttention.

    Reproduces open_clip's text encoding internals up to the final projection
    but skips the EOS-token pooling step that produces the single embedding.
    """

    def __init__(self, model_name='ViT-B-32', pretrained='laion2b_s34b_b79k'):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
        self.clip_model = clip_model
        self.tokenizer = open_clip.get_tokenizer(model_name)

        if hasattr(self.clip_model, 'visual'):
            del self.clip_model.visual

        for param in self.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def forward(self, text_list):
        device = next(self.parameters()).device
        tokens = self.tokenizer(text_list).to(device)
        cm = self.clip_model

        cast_dtype = cm.transformer.get_cast_dtype()
        x = cm.token_embedding(tokens).to(cast_dtype)
        x = x + cm.positional_embedding.to(cast_dtype)
        x = cm.transformer(x, attn_mask=cm.attn_mask)
        x = cm.ln_final(x)

        # Apply text projection to every token (CLIP normally applies it only to EOS)
        if cm.text_projection is not None:
            if isinstance(cm.text_projection, nn.Linear):
                x = cm.text_projection(x)
            else:
                x = x @ cm.text_projection

        # L2 normalize per token
        x = x / (x.norm(dim=-1, keepdim=True) + 1e-8)
        return x  # (B, T, 512)

In [39]:
class TextAugmentedDecoderFiLM(nn.Module):
    """UNetFormer decoder with FiLM fusion at each decoder stage.

    Identical to TextAugmentedDecoder except cross-attention is replaced with
    FiLMFusion. Takes a pooled text embedding (B, 512), same input as Phase 2.
    """

    def __init__(self, encoder_channels=(64, 128, 256, 512), decode_channels=64,
                 dropout=0.1, window_size=8, num_classes=7, text_dim=512):
        super().__init__()
        self.pre_conv = ConvBN(encoder_channels[-1], decode_channels, kernel_size=1)
        self.b4 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p3 = WF(encoder_channels[-2], decode_channels)
        self.b3 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p2 = WF(encoder_channels[-3], decode_channels)
        self.b2 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p1 = FeatureRefinementHead(encoder_channels[-4], decode_channels)
        self.up4 = nn.UpsamplingBilinear2d(scale_factor=4)
        self.up3 = nn.UpsamplingBilinear2d(scale_factor=2)
        self.aux_head = AuxHead(decode_channels, num_classes)
        self.segmentation_head = nn.Sequential(
            ConvBNReLU(decode_channels, decode_channels),
            nn.Dropout2d(p=dropout, inplace=True),
            Conv(decode_channels, num_classes, kernel_size=1),
        )

        # FiLM fusion at the same 4 stages as Phase 2 cross-attention
        self.text_fuse4 = FiLMFusion(decode_channels, text_dim)
        self.text_fuse3 = FiLMFusion(decode_channels, text_dim)
        self.text_fuse2 = FiLMFusion(decode_channels, text_dim)
        self.text_fuse1 = FiLMFusion(decode_channels, text_dim)

        self.init_weight()

    def forward(self, res1, res2, res3, res4, h, w, text_embed):
        if self.training:
            x = self.b4(self.pre_conv(res4)); x = self.text_fuse4(x, text_embed); h4 = self.up4(x)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_fuse3(x, text_embed); h3 = self.up3(x)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_fuse2(x, text_embed); h2 = x
            x = self.p1(x, res1); x = self.text_fuse1(x, text_embed)
            x = self.segmentation_head(x)
            x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)
            ah = h4 + h3 + h2; ah = self.aux_head(ah, h, w)
            return x, ah
        else:
            x = self.b4(self.pre_conv(res4)); x = self.text_fuse4(x, text_embed)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_fuse3(x, text_embed)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_fuse2(x, text_embed)
            x = self.p1(x, res1); x = self.text_fuse1(x, text_embed)
            x = self.segmentation_head(x)
            return F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)

    def init_weight(self):
        for m in self.children():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)


class UNetFormerCLIPFiLM(nn.Module):
    """UNetFormer + frozen CLIP text encoder + FiLM fusion."""

    def __init__(self, decode_channels=64, dropout=0.1, backbone_name='swsl_resnet18',
                 pretrained=True, window_size=8, num_classes=7):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, features_only=True, output_stride=32,
                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)
        encoder_channels = self.backbone.feature_info.channels()
        self.text_encoder = CLIPTextEncoder()  # pooled embedding, same as Phase 2
        self.decoder = TextAugmentedDecoderFiLM(encoder_channels, decode_channels, dropout,
                                                window_size, num_classes, text_dim=512)

    def forward(self, x, captions):
        h, w = x.size()[-2:]
        res1, res2, res3, res4 = self.backbone(x)
        text_embed = self.text_encoder(captions)
        if self.training:
            out, ah = self.decoder(res1, res2, res3, res4, h, w, text_embed)
            return out, ah
        else:
            return self.decoder(res1, res2, res3, res4, h, w, text_embed)

In [40]:
class TextAugmentedDecoderSeq(nn.Module):
    """UNetFormer decoder with multi-token cross-attention at each decoder stage."""

    def __init__(self, encoder_channels=(64, 128, 256, 512), decode_channels=64,
                 dropout=0.1, window_size=8, num_classes=7, text_dim=512):
        super().__init__()
        self.pre_conv = ConvBN(encoder_channels[-1], decode_channels, kernel_size=1)
        self.b4 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p3 = WF(encoder_channels[-2], decode_channels)
        self.b3 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p2 = WF(encoder_channels[-3], decode_channels)
        self.b2 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p1 = FeatureRefinementHead(encoder_channels[-4], decode_channels)
        self.up4 = nn.UpsamplingBilinear2d(scale_factor=4)
        self.up3 = nn.UpsamplingBilinear2d(scale_factor=2)
        self.aux_head = AuxHead(decode_channels, num_classes)
        self.segmentation_head = nn.Sequential(
            ConvBNReLU(decode_channels, decode_channels),
            nn.Dropout2d(p=dropout, inplace=True),
            Conv(decode_channels, num_classes, kernel_size=1),
        )

        self.text_attn4 = MultiTokenCrossAttention(decode_channels, text_dim)
        self.text_attn3 = MultiTokenCrossAttention(decode_channels, text_dim)
        self.text_attn2 = MultiTokenCrossAttention(decode_channels, text_dim)
        self.text_attn1 = MultiTokenCrossAttention(decode_channels, text_dim)

        self.init_weight()

    def forward(self, res1, res2, res3, res4, h, w, text_seq):
        if self.training:
            x = self.b4(self.pre_conv(res4)); x = self.text_attn4(x, text_seq); h4 = self.up4(x)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_attn3(x, text_seq); h3 = self.up3(x)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_attn2(x, text_seq); h2 = x
            x = self.p1(x, res1); x = self.text_attn1(x, text_seq)
            x = self.segmentation_head(x)
            x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)
            ah = h4 + h3 + h2; ah = self.aux_head(ah, h, w)
            return x, ah
        else:
            x = self.b4(self.pre_conv(res4)); x = self.text_attn4(x, text_seq)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_attn3(x, text_seq)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_attn2(x, text_seq)
            x = self.p1(x, res1); x = self.text_attn1(x, text_seq)
            x = self.segmentation_head(x)
            return F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)

    def init_weight(self):
        for m in self.children():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)


class UNetFormerCLIPSeq(nn.Module):
    """UNetFormer + frozen CLIP sequence text encoder + multi-token cross-attention."""

    def __init__(self, decode_channels=64, dropout=0.1, backbone_name='swsl_resnet18',
                 pretrained=True, window_size=8, num_classes=7):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, features_only=True, output_stride=32,
                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)
        encoder_channels = self.backbone.feature_info.channels()
        self.text_encoder = CLIPSequenceTextEncoder()
        self.decoder = TextAugmentedDecoderSeq(encoder_channels, decode_channels, dropout,
                                               window_size, num_classes, text_dim=512)

    def forward(self, x, captions):
        h, w = x.size()[-2:]
        res1, res2, res3, res4 = self.backbone(x)
        text_seq = self.text_encoder(captions)
        if self.training:
            out, ah = self.decoder(res1, res2, res3, res4, h, w, text_seq)
            return out, ah
        else:
            return self.decoder(res1, res2, res3, res4, h, w, text_seq)

In [41]:
# Sanity check on both fusion variants
for name, ctor in [('FiLM', UNetFormerCLIPFiLM), ('MultiToken', UNetFormerCLIPSeq)]:
    m = ctor(num_classes=NUM_CLASSES).to(device)
    total = sum(p.numel() for p in m.parameters()) / 1e6
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6
    with torch.no_grad():
        sample_imgs, _, sample_caps = next(iter(train_loader))
        sample_imgs = sample_imgs[:2].to(device)
        sample_caps = list(sample_caps[:2])
        m.eval()
        out = m(sample_imgs, sample_caps)
    print(f'{name:<10}: total {total:.2f}M, trainable {trainable:.2f}M, output {tuple(out.shape)}')
    del m
    torch.cuda.empty_cache()

FiLM      : total 75.45M, trainable 12.02M, output (2, 7, 256, 256)
MultiToken: total 75.37M, trainable 11.94M, output (2, 7, 256, 256)


In [50]:
# Train FiLM and multi-token variants on the Phase 2 best caption
film_factory       = lambda: UNetFormerCLIPFiLM(num_classes=NUM_CLASSES)
multitoken_factory = lambda: UNetFormerCLIPSeq(num_classes=NUM_CLASSES)

fusion_runs = {}

for label, factory, wandb_extra in [
    ('film',       film_factory,       {'fusion_type': 'film'}),
    ('multitoken', multitoken_factory, {'fusion_type': 'multi_token_cross_attention'}),
]:
    run_name = f'fusion_{label}_text_qwen3_4b'
    print(f'\n{"=" * 60}')
    print(f'Fusion: {label}')
    print(f'{"=" * 60}\n')

    hist, val_miou, ckpt = train_multimodal(
        caption_col=BEST_CAPTION,
        model_factory=factory,
        run_name=run_name,
        wandb_config={'experiment': 'ablation_3_fusion', **wandb_extra},
    )
    save_history(hist, run_name)
    fusion_runs[label] = {'history': hist, 'val_miou': val_miou, 'ckpt': ckpt,
                          'factory': factory}


Fusion: film



Training fusion_film_text_qwen3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.2698   0.5247   0.5183   0.7832   54.2s   BEST
    2   0.7711   0.4171   0.4888   0.7633   53.9s       
    3   0.6766   0.4368   0.5015   0.7501   53.9s       
    4   0.6154   0.4345   0.4748   0.7446   53.2s       
    5   0.5638   0.3491   0.5598   0.7846   54.6s   BEST
    6   0.5179   0.3901   0.5413   0.7482   52.8s       
    7   0.5072   0.3418   0.5635   0.7943   53.8s   BEST
    8   0.4779   0.2954   0.6288   0.8438   54.1s   BEST
    9   0.4450   0.3560   0.6194   0.8221   54.4s       
   10   0.4652   0.3081   0.6069   0.8403   54.1s       
   11   0.4247   0.2943   0.5961   0.8129   53.1s       
   12   0.4099   0.2789   0.6192   0.8436   54.1s       
   13   0.3848   0.2717   0.5935   0.8298   53.7s       
   14   0.3774   0.2936   0.6282   0.8417   54.2s       
   15   0.3665   0.3741   0.5803   0.

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▄▃▂▂▃▁▄▅▆▅▃▆▅▅▂█▅▇▅▆▇▇▆▆▇█▇▇▇▆
val_iou/Built-up,▆▁▁▂▆▅▃▆▇▆▆▄▃▆█▇█▆██▇██▇▇████▇
val_iou/Crop,▁▃▃▄▄▅▃▆▄▅▅▆▇▆▁▇▇█▇▆█▇████████
val_iou/Grass,▃▁▂▃▃▁▃▆▄▅▃▆▅▆▂▇▆▇▇▇▇▇▇▇█████▇
val_iou/Shrub,▃▃▁▁▂▁▃▅▃▆▄▅▃▄▄▄▆▃▇██▇▇▇██▇██▇
val_iou/Tree,▄▄▄▁▅▄▆▇▆▇▇▇▆▆▄▇█▇████████████
val_iou/Water,▁▁▄▁▅▆▇▇█▄▇▇▆█▇███████████████
+3,...



Best validation mIoU: 0.6919
Saved history to /content/drive/MyDrive/DI725_Project/results/fusion_film_text_qwen3_4b_history.json

Fusion: multitoken



Training fusion_multitoken_text_qwen3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.1574   0.5959   0.4721   0.7254   59.6s   BEST
    2   0.7514   0.4370   0.5339   0.7640   60.5s   BEST
    3   0.6510   0.3621   0.5805   0.8054   59.6s   BEST
    4   0.6072   0.8104   0.3445   0.6159   59.7s       
    5   0.5768   0.3503   0.6204   0.8259   59.9s   BEST
    6   0.5333   0.3660   0.5846   0.8078   60.3s       
    7   0.5040   0.3328   0.5820   0.8166   59.2s       
    8   0.4648   0.3373   0.5996   0.8221   59.3s       
    9   0.4542   0.3017   0.5982   0.8198   59.1s       
   10   0.4325   0.2976   0.5922   0.8099   59.4s       
   11   0.3972   0.2690   0.6325   0.8391   60.2s   BEST
   12   0.3838   0.2793   0.6184   0.8424   59.0s       
   13   0.3823   0.2569   0.6455   0.8535   60.5s   BEST
   14   0.3528   0.2572   0.6386   0.8380   59.1s       
   15   0.3449   0.2525   0.619

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▃▆▁▅▄▅▆▅▅▆▆▆▅▅▆▇▇▇▆▇█▇▇▇██▇▇█
val_iou/Built-up,▃▅▆▁█▇▃▆▅▆█▆▆█▅▇▇▇▇▇▆█▇▇██▇█▇█
val_iou/Crop,▁▂▄▁▄▄▅▃▅▅▆▆▆▆▆▄▆▇▇▇██▇▇██████
val_iou/Grass,▂▃▄▁▅▄▅▅▅▄▆▆▇▆▆▆▇▇▇▇▇█▇▇██████
val_iou/Shrub,▁▁▂▁▄▄▃▅▃▃▃▅▅▄▆▆▅▅▆▇█▆▆▇▆█▇▇█▇
val_iou/Tree,▆▆▇▁▇▇▇▇▇▇▇▇█▇█▇██████████████
val_iou/Water,▅▆▇▁█▇▇▇▇▇█▇██▇███████████████
+3,...



Best validation mIoU: 0.6949
Saved history to /content/drive/MyDrive/DI725_Project/results/fusion_multitoken_text_qwen3_4b_history.json


In [51]:
# Test set evaluation for both fusion variants
fusion_test = {}

print('Evaluating two fusion variants on text_qwen3-4b...\n')
print(f"{'Fusion':<12} {'test mIoU':>11} {'test OA':>10}")
print('-' * 36)

for label, run_info in fusion_runs.items():
    class_ious, miou, oa, _ = evaluate_checkpoint(
        caption_col=BEST_CAPTION,
        ckpt_path=run_info['ckpt'],
        model_factory=run_info['factory'],
    )
    fusion_test[label] = {'class_ious': class_ious, 'miou': miou, 'oa': oa}
    print(f'{label:<12} {miou:>11.4f} {oa:>10.4f}')

Evaluating two fusion variants on text_qwen3-4b...

Fusion         test mIoU    test OA
------------------------------------
film              0.6904     0.8871
multitoken        0.6915     0.8881


In [52]:
# Three-way per-class comparison: FiLM vs Pooled (Phase 2) vs MultiToken
print(f'Fusion mechanism comparison on {BEST_CAPTION}:\n')
print(f"{'Class':<10} {'FiLM':>10} {'Pooled':>10} {'MultiTok':>10} {'ΔFiLM':>9} {'ΔMT':>9}")
print('-' * 64)

pooled_ious = MULTIMODAL_IOUS  # Phase 2 pooled cross-attention

for i, name in enumerate(CLASS_NAMES):
    f_iou = fusion_test['film']['class_ious'][i]
    p_iou = pooled_ious[i]
    m_iou = fusion_test['multitoken']['class_ious'][i]
    if any(v is None or (isinstance(v, float) and np.isnan(v)) for v in [f_iou, p_iou, m_iou]):
        print(f'{name:<10} {"N/A":>10} {"N/A":>10} {"N/A":>10}')
        continue
    print(f'{name:<10} {f_iou:>10.4f} {p_iou:>10.4f} {m_iou:>10.4f} '
          f'{f_iou-p_iou:>+9.4f} {m_iou-p_iou:>+9.4f}')

print('-' * 64)
miou_film = fusion_test['film']['miou']
miou_mt   = fusion_test['multitoken']['miou']
print(f'{"mIoU":<10} {miou_film:>10.4f} {MULTIMODAL_MIOU:>10.4f} {miou_mt:>10.4f} '
      f'{miou_film - MULTIMODAL_MIOU:>+9.4f} {miou_mt - MULTIMODAL_MIOU:>+9.4f}')

print()
print(f'Relative change vs Phase 2 pooled cross-attention (mIoU = {MULTIMODAL_MIOU:.4f}):')
print(f'  FiLM       : {(miou_film - MULTIMODAL_MIOU) / MULTIMODAL_MIOU * 100:+.2f}%')
print(f'  MultiToken : {(miou_mt   - MULTIMODAL_MIOU) / MULTIMODAL_MIOU * 100:+.2f}%')

# Best fusion mechanism (used to select the final-model configuration in Section 9)
best_fusion_miou = max(MULTIMODAL_MIOU, miou_film, miou_mt)
if best_fusion_miou == miou_mt:
    best_fusion = 'multitoken'
elif best_fusion_miou == miou_film:
    best_fusion = 'film'
else:
    best_fusion = 'pooled'
print(f'\nBest fusion: {best_fusion} (test mIoU = {best_fusion_miou:.4f})')

Fusion mechanism comparison on text_qwen3-4b:

Class            FiLM     Pooled   MultiTok     ΔFiLM       ΔMT
----------------------------------------------------------------
Tree           0.8745     0.8758     0.8729   -0.0013   -0.0029
Shrub          0.3090     0.3261     0.3194   -0.0171   -0.0068
Grass          0.8021     0.8057     0.8035   -0.0036   -0.0021
Crop           0.8107     0.8159     0.8189   -0.0052   +0.0030
Built-up       0.5735     0.5893     0.5919   -0.0158   +0.0026
Barren         0.5748     0.5665     0.5670   +0.0083   +0.0005
Water          0.8879     0.8994     0.8673   -0.0115   -0.0321
----------------------------------------------------------------
mIoU           0.6904     0.6970     0.6915   -0.0066   -0.0054

Relative change vs Phase 2 pooled cross-attention (mIoU = 0.6970):
  FiLM       : -0.95%
  MultiToken : -0.78%

Best fusion: pooled (test mIoU = 0.6970)


In [53]:
# Persist Ablation 3 results
ablation3_results = {
    'experiment':  'fusion_mechanism',
    'caption_col': BEST_CAPTION,
    'phase2_pooled_cross_attention': {
        'miou': MULTIMODAL_MIOU, 'oa': MULTIMODAL_OA, 'class_ious': MULTIMODAL_IOUS,
    },
    'phase3_film': {
        'miou':       fusion_test['film']['miou'],
        'oa':         fusion_test['film']['oa'],
        'class_ious': [float(x) if not np.isnan(x) else None
                       for x in fusion_test['film']['class_ious']],
    },
    'phase3_multitoken': {
        'miou':       fusion_test['multitoken']['miou'],
        'oa':         fusion_test['multitoken']['oa'],
        'class_ious': [float(x) if not np.isnan(x) else None
                       for x in fusion_test['multitoken']['class_ious']],
    },
    'best_fusion': best_fusion,
}

with open(RESULTS_DIR / 'ablation3_fusion.json', 'w') as f:
    json.dump(ablation3_results, f, indent=2)
print(f"Saved to {RESULTS_DIR / 'ablation3_fusion.json'}")

Saved to /content/drive/MyDrive/DI725_Project/results/ablation3_fusion.json


**Observations.**

Pooled cross-attention from Phase 2 outperforms both alternatives on test mIoU. FiLM reaches 0.6904 (−0.95% relative to pooled) and multi-token cross-attention reaches 0.6915 (−0.78%).

Per-class IoU for FiLM vs pooled is lower on six of seven classes, with the largest drops on Shrub (−0.0171), Built-up (−0.0158), and Water (−0.0115). FiLM gains only on Barren (+0.0083).

Per-class IoU for multi-token vs pooled is lower on five of seven classes, with the largest drop on Water (−0.0321). Multi-token gains modestly on Crop (+0.0030) and Built-up (+0.0026).

## 9. Additivity check — do the encoder and loss gains stack?

Ablation 2 (CE+Lovász on `text_qwen3-4b`) produced the strongest single-component improvement: test mIoU 0.7387 (+5.99% vs Phase 2 multimodal). Ablation 1 (RemoteCLIP on the same caption) produced a smaller improvement of +0.0032 (+0.46%).

This section tests whether combining the two improvements produces an additive gain (predicted test mIoU ≈ 0.7420) or whether there is an interaction between the loss and the encoder.

Configuration trained: **RemoteCLIP text encoder + CE+Lovász loss + pooled cross-attention, on `text_qwen3-4b`**. This is identical to the Ablation 2 winning configuration except that the text encoder is swapped from CLIP to RemoteCLIP.

The best Phase 3 model is selected after this analysis based on observed test mIoU.

In [60]:
# Combined model factory: RemoteCLIP text encoder + UNetFormer + pooled cross-attention
combined_model_factory = lambda: UNetFormerRemoteCLIP(num_classes=NUM_CLASSES)

# Loss: CE + Lovász (the winning composite loss from Ablation 2)
combined_criterion = CompositeLoss(
    class_weights_tensor,
    LovaszSoftmaxLoss(),
    alpha=LOVASZ_ALPHA,
)

print('Combined model configuration:')
print(f'  Text encoder    : RemoteCLIP (ViT-B-32)')
print(f'  Fusion          : Pooled cross-attention (Phase 2)')
print(f'  Loss            : CE + Lovász-Softmax (alpha={LOVASZ_ALPHA})')
print(f'  Caption variant : {BEST_CAPTION}')
print(f'  Training recipe : {NUM_EPOCHS} epochs, AdamW lr={LR}, weight decay {WEIGHT_DECAY}')

Combined model configuration:
  Text encoder    : RemoteCLIP (ViT-B-32)
  Fusion          : Pooled cross-attention (Phase 2)
  Loss            : CE + Lovász-Softmax (alpha=0.5)
  Caption variant : text_qwen3-4b
  Training recipe : 30 epochs, AdamW lr=0.0006, weight decay 0.01


In [61]:
# Train the combined model (additivity test)
print(f'{"=" * 60}')
print('Training combined model: RemoteCLIP + CE+Lovász + pooled, on text_qwen3-4b')
print(f'{"=" * 60}\n')

hist_combined, val_miou_combined, ckpt_combined = train_multimodal(
    caption_col=BEST_CAPTION,
    model_factory=combined_model_factory,
    criterion=combined_criterion,
    run_name='additivity_remoteclip_lovasz',
    wandb_config={
        'experiment':      'additivity_check',
        'text_encoder':    'remoteclip',
        'loss_type':       'ce_lovasz',
        'composite_alpha': LOVASZ_ALPHA,
        'fusion_type':     'pooled_cross_attention',
    },
)
save_history(hist_combined, 'additivity_remoteclip_lovasz')

Training combined model: RemoteCLIP + CE+Lovász + pooled, on text_qwen3-4b

RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


Training additivity_remoteclip_lovasz for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   0.9463   0.4720   0.5398   0.7677   69.1s   BEST
    2   0.7142   0.4524   0.5707   0.7894   69.0s   BEST
    3   0.6698   0.4298   0.6268   0.8266   68.4s   BEST
    4   0.6329   0.4293   0.5450   0.7836   68.8s       
    5   0.5955   0.3922   0.6215   0.8342   67.7s       
    6   0.5792   0.4059   0.6528   0.8543   68.5s   BEST
    7   0.5624   0.3894   0.6295   0.8447   67.8s       
    8   0.5403   0.3939   0.5942   0.8241   67.9s       
    9   0.5301   0.3615   0.6766   0.8752   68.9s   BEST
   10   0.5159   0.3742   0.6613   0.8562   68.2s       
   11   0.5024   0.3425   0.6818   0.8730   68.0s   BEST
   12   0.4932   0.3535   0.6616   0.8523   68.2s       
   13   0.4799   0.3520   0.6522   0.8628   68.7s       
   14   0.4683   0.3465   0.6879   0.8731   69.5s   BEST
   15   0.4588   0.3452   0.6687  

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▂▄▂▄▅▆▃▆▅▆▄▅▆▅▇▆▇█▇▇▇█▇▇█████
val_iou/Built-up,▁▃▆▁▅▆▅▅▆▇▆▇▆▇▆▆▇▇███▇▇█▇█████
val_iou/Crop,▁▃▃▁▄▃▅▂▅▅▅▄▅▆▆▇▅▇▇▆▇▇████████
val_iou/Grass,▁▃▃▁▅▅▅▄▇▅▆▅▆▇▆▇▇▇▇▇█▇████████
val_iou/Shrub,▁▁▂▂▃▅▂▃▆▄▆▄▄▅▄▆▅▅▄▇▅▇▇▇▇▇█▇██
val_iou/Tree,▁▁▄▃▅▅▅▅▇▆▇▆▆▆▆▇▇▇▇▇▇█████████
val_iou/Water,▄▅▇▂▅▆▄▁▆▆▇▇▄██▇▇▇████████████
+3,...



Best validation mIoU: 0.7344
Saved history to /content/drive/MyDrive/DI725_Project/results/additivity_remoteclip_lovasz_history.json


In [62]:
# Test set evaluation
class_ious_combined, miou_combined, oa_combined, _ = evaluate_checkpoint(
    caption_col=BEST_CAPTION,
    ckpt_path=ckpt_combined,
    model_factory=combined_model_factory,
    criterion=combined_criterion,
)

print(f'Combined model on test set:')
print(f'  test mIoU = {miou_combined:.4f}')
print(f'  test OA   = {oa_combined:.4f}')

RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>
Combined model on test set:
  test mIoU = 0.7365
  test OA   = 0.9041


In [65]:
# Additivity analysis
remoteclip_alone_miou = remoteclip_test[BEST_CAPTION]['miou']
lovasz_alone_miou     = loss_test['ce_lovasz']['miou']

# Single-component deltas vs Phase 2 anchor
delta_remoteclip = remoteclip_alone_miou - MULTIMODAL_MIOU
delta_lovasz     = lovasz_alone_miou     - MULTIMODAL_MIOU
delta_combined   = miou_combined         - MULTIMODAL_MIOU

# Interaction: how much the combined gain differs from the sum of single gains
# Negative = sub-additive (combined < sum); positive = super-additive
interaction_term = delta_combined - (delta_remoteclip + delta_lovasz)

print('Additivity analysis:\n')
print(f"{'Configuration':<32} {'test mIoU':>11} {'Δ vs Phase 2':>14}")
print('-' * 60)
print(f"{'Phase 2 anchor (CLIP + wCE)':<32} {MULTIMODAL_MIOU:>11.4f} {'—':>14}")
print(f"{'RemoteCLIP alone':<32} {remoteclip_alone_miou:>11.4f} {delta_remoteclip:>+14.4f}")
print(f"{'CE+Lovász alone':<32} {lovasz_alone_miou:>11.4f} {delta_lovasz:>+14.4f}")
print(f"{'Combined (RemoteCLIP + Lovász)':<32} {miou_combined:>11.4f} {delta_combined:>+14.4f}")
print('-' * 60)
print(f"{'Interaction term':<32} {'':>11} {interaction_term:>+14.4f}")

print()
print(f'Relative improvement vs Phase 2 (mIoU = {MULTIMODAL_MIOU:.4f}):')
print(f'  RemoteCLIP alone : {delta_remoteclip / MULTIMODAL_MIOU * 100:+.2f}%')
print(f'  CE+Lovász alone  : {delta_lovasz     / MULTIMODAL_MIOU * 100:+.2f}%')
print(f'  Combined         : {delta_combined   / MULTIMODAL_MIOU * 100:+.2f}%')

# Best Phase 3 configuration (selected from the three Phase 3 results above)
phase3_results = {
    'remoteclip_alone': remoteclip_alone_miou,
    'ce_lovasz_alone':  lovasz_alone_miou,
    'combined':         miou_combined,
}
best_phase3 = max(phase3_results, key=phase3_results.get)
best_phase3_miou = phase3_results[best_phase3]
print(f'\nBest Phase 3 configuration: {best_phase3} (test mIoU = {best_phase3_miou:.4f})')

Additivity analysis:

Configuration                      test mIoU   Δ vs Phase 2
------------------------------------------------------------
Phase 2 anchor (CLIP + wCE)           0.6970              —
RemoteCLIP alone                      0.7002        +0.0032
CE+Lovász alone                       0.7387        +0.0418
Combined (RemoteCLIP + Lovász)        0.7365        +0.0396
------------------------------------------------------------
Interaction term                                    -0.0054

Relative improvement vs Phase 2 (mIoU = 0.6970):
  RemoteCLIP alone : +0.46%
  CE+Lovász alone  : +5.99%
  Combined         : +5.68%

Best Phase 3 configuration: ce_lovasz_alone (test mIoU = 0.7387)


In [66]:
# Persist additivity check results
additivity_results = {
    'experiment':  'additivity_check',
    'caption_col': BEST_CAPTION,
    'config': {
        'text_encoder':    'remoteclip',
        'fusion_type':     'pooled_cross_attention',
        'loss_type':       'ce_lovasz',
        'composite_alpha': LOVASZ_ALPHA,
    },
    'phase2_anchor': {
        'miou': MULTIMODAL_MIOU, 'oa': MULTIMODAL_OA, 'class_ious': MULTIMODAL_IOUS,
    },
    'remoteclip_alone': {
        'miou': remoteclip_alone_miou,
        'oa':   remoteclip_test[BEST_CAPTION]['oa'],
        'class_ious': remoteclip_test[BEST_CAPTION]['class_ious'],
    },
    'ce_lovasz_alone': {
        'miou': lovasz_alone_miou,
        'oa':   loss_test['ce_lovasz']['oa'],
        'class_ious': loss_test['ce_lovasz']['class_ious'],
    },
    'combined': {
        'miou':       miou_combined,
        'oa':         oa_combined,
        'class_ious': [float(x) if not np.isnan(x) else None for x in class_ious_combined],
    },
    'additivity': {
        'delta_remoteclip':   delta_remoteclip,
        'delta_lovasz':       delta_lovasz,
        'delta_combined':     delta_combined,
        'predicted_additive': predicted_additive,
        'interaction_term':   interaction_term,
    },
    'best_phase3': best_phase3,
    'best_phase3_miou': best_phase3_miou,
}

with open(RESULTS_DIR / 'additivity_check.json', 'w') as f:
    json.dump(additivity_results, f, indent=2)
print(f"Saved to {RESULTS_DIR / 'additivity_check.json'}")

Saved to /content/drive/MyDrive/DI725_Project/results/additivity_check.json


**Observations.**

The encoder and loss gains do not stack additively. CE+Lovász alone reaches test mIoU 0.7387 (+5.99% vs Phase 2). RemoteCLIP alone reaches 0.7002 (+0.46%). The combined model (RemoteCLIP + CE+Lovász) reaches 0.7365 (+5.68%), which is 0.0022 below CE+Lovász alone.

The interaction term is −0.0054, indicating sub-additive composition: combining the two improvements gives less than the sum of their individual gains (the additive prediction would have been 0.7420). The combined model also underperforms CE+Lovász alone, meaning RemoteCLIP does not contribute a positive gain in the presence of CE+Lovász.

Per-class, the combined model performs comparably to or slightly below CE+Lovász alone on most classes, with no class showing a meaningful gain from the encoder swap when paired with the Lovász loss.

The best Phase 3 model is therefore **CE+Lovász** (with CLIP, pooled cross-attention) at test mIoU 0.7387. The combined model is reported as a finding about gain composition, not as the selected configuration.